# GigaHands · 두 손 3D 포즈 학습

현재 프로젝트의 **시간(초) 기준 이벤트 모델**을 포함한 독립 실행 노트북입니다. 입력은 실제 도착한 카메라 프레임(이벤트) 목록이고, 임의의 query 시각마다 `[2,21,3]`을 출력합니다. 설정 셀의 `ARCH`로 모델을 고릅니다: 기본 `'direct'`(HandDirect: 신경망만으로 좌표를 직접 출력, 기하 계산 없음) 또는 `'lite'`(HandLite: 삼각측량 anchor + 신경망 보정, 비교용). 둘 다 카메라당 최근 이벤트 8개만 쓰는 고정 크기 모델이라 ncnn/ONNX로 라즈베리 파이에 배포합니다. 별도 저장소 복제가 필요하지 않습니다. 모든 범위가 초 단위라 출력 fps에 묶이지 않으며 실제 파라미터 수는 아래 확인 셀에서 표시합니다. 카메라 보정용 보조 정답 `[3,6]`은 NPZ의 실제·명목 카메라 값에서 계산해 손실에만 사용합니다.

## 먼저 할 일
1. Colab에서 이 `.ipynb`를 엽니다.
2. **런타임 → 런타임 유형 변경 → GPU**를 선택합니다.
3. PC의 `exports/gigahands_pi3_overlap.zip`을 Google Drive의 `MyDrive/GigaHands/`에 올립니다. ZIP 안에는 `manifest.jsonl`, `train/`, `val/`가 있어야 합니다. 원본 GigaHands tar.gz가 아닙니다.
4. 아래 설정 셀의 경로와 실행 이름을 확인한 뒤 위에서부터 실행합니다.

데이터: 원본 클립 12,673개, 학습 9,628개/검증 1,078개, 참가자 44명/6명. ZIP 약 6.92GB, 압축 해제 후 약 7.00GB입니다. Colab GPU 종류·사용 시간은 가용성에 따라 달라집니다. Drive에 많은 작은 파일을 직접 반복 읽는 대신 ZIP을 런타임 로컬로 복사합니다. [Colab 공식 FAQ](https://research.google.com/colaboratory/faq.html)

기존 이전 아키텍처의 smoke 가중치는 사용하지 않습니다. 새 모델을 처음부터 학습하며, 중단 후에는 이 노트북이 저장한 같은 실행의 가중치만 재개합니다.


In [ ]:
# 1. 환경 확인 — Colab에 설치된 CUDA용 PyTorch를 그대로 사용합니다.
import sys, os, json, hashlib, shutil, subprocess, importlib.util
from pathlib import Path, PurePosixPath
import torch

assert tuple(map(int, torch.__version__.split('+')[0].split('.')[:2])) >= (2, 5), 'PyTorch 2.5 이상이 필요합니다.'
assert torch.cuda.is_available(), 'Colab 런타임 유형을 GPU로 바꾸고 다시 실행하세요.'
missing = [name for name in ('numpy', 'matplotlib', 'tqdm') if importlib.util.find_spec(name) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)
import numpy as np
from tqdm.auto import tqdm
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('GPU memory: %.1f GB' % (torch.cuda.get_device_properties(0).total_memory / 1e9))
WORK_DIR = Path('/content/gigahands_training')
WORK_DIR.mkdir(exist_ok=True)
for package in ('hand_tracking', 'training'):
    (WORK_DIR / package).mkdir(exist_ok=True)
os.chdir(WORK_DIR)
if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))


In [ ]:
# 2. Drive 연결
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 3. 사용자 설정
DRIVE_ZIP = Path('/content/drive/MyDrive/GigaHands/gigahands_pi3_overlap.zip')
DRIVE_RUNS = Path('/content/drive/MyDrive/GigaHands/runs')
RUN_NAME = 'event_direct_fusion2_e60_v2'  # 이전 노트북의 실행과 소스가 달라 새 이름으로 학습
EPOCHS = 60
BATCH_SIZE = 64                   # × TRAIN_QUERIES = 스텝당 query 256개; 메모리 부족이면 새 실행 이름으로 32
LEARNING_RATE = 6e-4              # 배치 16일 때 3e-4
WINDOWS_PER_CLIP = 16             # epoch마다 클립당 무작위 구간; 0이면 전체
TRAIN_QUERIES = 4                 # 학습은 윈도의 마지막 query 4개만(평가 지표는 원래 마지막 query); 0이면 16개 전부
COMPILE = False                   # True: torch.compile로 신경망 가속 시도(미검증). 첫 배치가 느리고, 오류면 False로
VAL_WINDOWS_PER_CLIP = 16         # 매 epoch 고정 검증 구간; 0이면 전체
WORKERS = 2
SEED = 42
# 'direct': HandDirect — 신경망만(이벤트 encoder → 토큰 fusion → 관절 decoder), 좌표 직접 출력
# 'lite': HandLite — 삼각측량 anchor + 신경망 보정(비교용)
ARCH = 'direct'
MODEL_OPTIONS = {
    'direct': {'--dim': 64, '--heads': 4, '--fusion-blocks': 2, '--blocks': 2, '--slots-per-camera': 8,
               '--event-span-s': 0.5},
    'lite': {'--dim': 64, '--heads': 4, '--blocks': 2, '--slots-per-camera': 8,   # 카메라당 최근 이벤트 8개
             '--event-span-s': 0.5, '--anchor-lookback-s': 0.2,
             '--ray-outlier-ratio': 6.0},   # 이상치 ray 제거(가짜 손 검출·좌우 뒤바뀜); 0이면 끔
}[ARCH]
DROPOUT = 0.1
CALIBRATION_WEIGHT = 0.01         # 카메라 보정 보조 손실 가중치(lite만; direct는 보정 출력 없음)
MODEL_ARGS = ['--arch', ARCH, '--dropout', DROPOUT] + [value for item in MODEL_OPTIONS.items() for value in item]

assert RUN_NAME and Path(RUN_NAME).name == RUN_NAME and RUN_NAME not in ('.', '..')
assert EPOCHS > 0 and BATCH_SIZE > 0 and WORKERS >= 0
assert LEARNING_RATE > 0 and WINDOWS_PER_CLIP >= 0 and VAL_WINDOWS_PER_CLIP >= 0 and TRAIN_QUERIES >= 0
assert DRIVE_ZIP.is_file(), f'Drive에 데이터 ZIP을 올리고 경로를 확인하세요: {DRIVE_ZIP}'
LOCAL_ZIP = Path('/content/gigahands_pi3_overlap.zip')
EXTRACT_DIR = Path('/content/gigahands_dataset')
CACHE_DIR = Path('/content/gigahands_cache')   # 첫 epoch에 전처리한 클립(압축 없음); 이후 epoch는 여기서 읽음
EXPERIMENT_DIR = DRIVE_RUNS / RUN_NAME
RUN_DIR = EXPERIMENT_DIR / 'checkpoints'
print('학습 결과:', RUN_DIR)


## 데이터 복사 및 검사
처음 실행할 때 데이터 ZIP을 복사·해제합니다. 연결이 끊겨 런타임 로컬 파일이 사라지면 이 셀을 다시 실행하세요. Drive의 원본 ZIP과 체크포인트는 보존됩니다. 진행 막대가 표시됩니다.


In [ ]:
# 4. ZIP 로컬 복사 + 안전한 압축 해제 (기존 부분 해제는 다시 완료)
import zipfile, stat
EXPECTED_MANIFEST_SHA256 = 'b4e950b05438ffb5ad7ee846918cc553dfa1d9ff832386ce5b527003916816fc'
zip_size = DRIVE_ZIP.stat().st_size
if not LOCAL_ZIP.is_file() or LOCAL_ZIP.stat().st_size != zip_size:
    assert shutil.disk_usage('/content').free > zip_size + 7001743710 + 1_000_000_000, '로컬 디스크 공간이 부족합니다.'
    partial = LOCAL_ZIP.with_suffix('.zip.part')
    with DRIVE_ZIP.open('rb') as src, partial.open('wb') as dst, tqdm(total=zip_size, unit='B', unit_scale=True, desc='Copy ZIP') as bar:
        while chunk := src.read(8 * 1024 * 1024):
            dst.write(chunk)
            bar.update(len(chunk))
    partial.replace(LOCAL_ZIP)

EXTRACT_DIR.mkdir(exist_ok=True)
ready = EXTRACT_DIR / '.ready.json'
reuse = ready.is_file() and json.loads(ready.read_text()).get('manifest_sha256') == EXPECTED_MANIFEST_SHA256
if not reuse:
    with zipfile.ZipFile(LOCAL_ZIP) as archive:
        infos = archive.infolist()
        assert shutil.disk_usage('/content').free > sum(i.file_size for i in infos) + 1_000_000_000, '압축 해제 공간이 부족합니다.'
        seen = set()
        for info in infos:
            relative = PurePosixPath(info.filename.replace('\\', '/'))
            target = (EXTRACT_DIR / str(relative)).resolve()
            if relative.is_absolute() or '..' in relative.parts or not target.is_relative_to(EXTRACT_DIR.resolve()):
                raise ValueError(f'잘못된 ZIP 경로: {info.filename}')
            if stat.S_ISLNK(info.external_attr >> 16) or str(relative) in seen:
                raise ValueError(f'링크 또는 중복 ZIP 경로: {info.filename}')
            seen.add(str(relative))
        for info in tqdm(infos, desc='Extract ZIP'):
            relative = PurePosixPath(info.filename.replace('\\', '/'))
            target = EXTRACT_DIR / str(relative)
            if info.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(info) as src, target.open('wb') as dst:
                shutil.copyfileobj(src, dst, length=4 * 1024 * 1024)

manifests = list(EXTRACT_DIR.rglob('manifest.jsonl'))
assert len(manifests) == 1, 'manifest.jsonl이 하나인 데이터 ZIP이 필요합니다.'
DATA_DIR = manifests[0].parent
manifest_hash = hashlib.sha256(manifests[0].read_bytes()).hexdigest()
assert manifest_hash == EXPECTED_MANIFEST_SHA256, '이 노트북에 연결된 균등 데이터셋과 ZIP이 다릅니다.'
ready.write_text(json.dumps({'manifest_sha256': manifest_hash}))
print('DATA_DIR:', DATA_DIR)


## 모델 소스
아래 23개 셀은 노트북 생성 시점의 모델·로더·학습·평가·추론·ONNX 내보내기 코드를 로컬에 씁니다. 보통 수정할 필요가 없습니다. 보정 보조 정답은 회전 벡터 3개(라디안)와 위치 차이 3개(월드 단위)이며, 실제 카메라 roll=0°라도 월드 축으로 표현한 회전 보정 벡터의 세 번째 성분이 0일 필요는 없습니다. 정답은 모델 forward 입력이 아닌 보조 loss에만 전달합니다.


In [ ]:
%%writefile /content/gigahands_training/training/__init__.py
"""Command-line entry points: python -m training.<train|evaluate|predict|export>."""


In [ ]:
%%writefile /content/gigahands_training/training/train.py
"""Train the GigaHands time-based (camera event) pose model (default HandDirect, --arch)."""
import argparse
import hashlib
import json
import math
from pathlib import Path
import torch
from torch.utils.data import DataLoader
from hand_tracking.checkpoints import build_model, restore_training, save_checkpoint, training_checkpoint
from hand_tracking.config import ARCHITECTURES, SamplingConfig, add_model_arguments, model_config_from_args
from hand_tracking.data import HandWindows
from hand_tracking.engine import run_epoch

# Settings that must be > 0 and >= 0; the model's own fields are checked by its config.
POSITIVE = ('epochs', 'batch_size', 'threads', 'lr')
NONNEGATIVE = ('workers', 'seed', 'max_clips', 'max_train_batches', 'max_val_batches', 'bone_weight', 'train_queries',
               'weight_decay', 'calibration_weight', 'world_weight')
# Settings that change speed, not results: a resume may differ in them.
RUNTIME = ('workers', 'threads', 'cache_dir')


def build_parser(argv=None):
    """The model flags depend on --arch, so it is read first."""
    first = argparse.ArgumentParser(add_help=False)
    first.add_argument('--arch', choices=sorted(ARCHITECTURES), default='direct')
    architecture = first.parse_known_args(argv)[0].arch
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument('--arch', choices=sorted(ARCHITECTURES), default='direct', help='Model (see hand_tracking.checkpoints.MODELS)')
    p.add_argument('--data', default='exports/gigahands_balanced_5gb')
    p.add_argument('--output', default='runs/hand_direct')
    p.add_argument('--epochs', type=int, default=60)
    p.add_argument('--batch-size', type=int, default=64)
    add_model_arguments(p, ARCHITECTURES[architecture])
    p.add_argument('--max-events', type=int, default=SamplingConfig.max_events,
                   help='Event slots per window (oldest dropped beyond)')
    p.add_argument('--calibration-weight', type=float, default=.01, help='Auxiliary loss on the simulator camera error')
    p.add_argument('--target-frame', choices=('rig', 'world'), default='rig',
                   help="Pose targets in the placed rig's frame (rig-wide error removed) or the true world")
    p.add_argument('--world-weight', type=float, default=.1,
                   help='Extra weight of the world-frame pose loss (weak pull toward true world coordinates)')
    p.add_argument('--lr', type=float, default=6e-4)
    p.add_argument('--weight-decay', type=float, default=.01)
    p.add_argument('--bone-weight', type=float, default=.1)
    p.add_argument('--windows-per-clip', type=int, default=16)
    p.add_argument('--train-queries', type=int, default=4,
                   help="Train on each window's last N queries only (0: all); validation keeps all")
    p.add_argument('--val-windows-per-clip', type=int, default=16)
    p.add_argument('--workers', type=int, default=0)
    p.add_argument('--cache-dir', default='',
                   help='Folder for uncompressed preprocessed clips, written on first read (about 2x the data size)')
    p.add_argument('--threads', type=int, default=4)
    p.add_argument('--compile', action='store_true',
                   help="torch.compile the networks in place (checkpoints unchanged); try on GPU, untested on CPU")
    p.add_argument('--seed', type=int, default=42)
    p.add_argument('--device', choices=['auto', 'cpu', 'cuda'], default='auto')
    p.add_argument('--max-clips', type=int, default=0, help='Development subset per split; 0 means all')
    p.add_argument('--max-train-batches', type=int, default=0)
    p.add_argument('--max-val-batches', type=int, default=0)
    p.add_argument('--resume', action='store_true', help='Resume output/last.pt with identical settings')
    return p


def parse_settings(parser, argv=None):
    """Arguments plus validated model and sampling configs; errors exit through the parser."""
    args = parser.parse_args(argv)
    if min(getattr(args, name) for name in POSITIVE) <= 0 or not math.isfinite(args.lr):
        parser.error('Invalid training settings')
    if min(getattr(args, name) for name in NONNEGATIVE) < 0:
        parser.error('Settings must be nonnegative')
    try:
        config = model_config_from_args(args, ARCHITECTURES[args.arch])
        return args, config, SamplingConfig(max_events=args.max_events, target_frame=args.target_frame)
    except ValueError as error:
        parser.error(str(error))


def select_device(name):
    if name == 'auto':
        return torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if name == 'cuda' and not torch.cuda.is_available():
        raise RuntimeError('CUDA is unavailable in this PyTorch installation')
    return torch.device(name)


def record_run(parser, args, output):
    """Write run.json, or on resume check it matches; returns the recorded options."""
    options = vars(args).copy()
    options.pop('resume')
    options['manifest_sha256'] = hashlib.sha256((Path(args.data)/'manifest.jsonl').read_bytes()).hexdigest()
    path = output/'run.json'
    if args.resume:
        previous = json.loads(path.read_text())
        # Flags added after that run was started match when they are at their defaults.
        previous.update({k: v for k, v in options.items() if k not in previous and v == parser.get_default(k)})
        if ({k: v for k, v in previous.items() if k not in RUNTIME}
                != {k: v for k, v in options.items() if k not in RUNTIME}):
            raise ValueError('Resume settings/dataset differ')
    else:
        path.write_text(json.dumps(options, indent=2), encoding='utf-8')
    return options


def compile_networks(model):
    """Compile each learned network in place (nn.Module.compile keeps parameter names, so
    checkpoints load uncompiled). The geometry around them stays eager: it has data-dependent
    shapes (slot counts, trimmed padding). Shapes vary per batch, hence dynamic=True."""
    for module in model.children():
        module.compile(dynamic=True)


def build_loaders(args, config, sampling, device):
    train = HandWindows(args.data, 'train', windows_per_clip=args.windows_per_clip, seed=args.seed,
                        max_clips=args.max_clips,
                        context_s=config.context_s, max_events=sampling.max_events, target_frame=sampling.target_frame,
                        queries=args.train_queries, cache=args.cache_dir or None, seen_span_s=config.event_span_s)
    val = HandWindows(args.data, 'val', windows_per_clip=args.val_windows_per_clip, seed=args.seed,
                      max_clips=args.max_clips, context_s=config.context_s, max_events=sampling.max_events,
                      target_frame=sampling.target_frame, cache=args.cache_dir or None,
                      seen_span_s=config.event_span_s)
    return [DataLoader(dataset, batch_size=args.batch_size, num_workers=args.workers,
                       pin_memory=device.type == 'cuda') for dataset in (train, val)]


def main(argv=None):
    parser = build_parser(argv)
    args, config, sampling = parse_settings(parser, argv)
    torch.set_num_threads(args.threads)
    torch.manual_seed(args.seed)
    device = select_device(args.device)
    output = Path(args.output)
    output.mkdir(parents=True, exist_ok=args.resume)
    options = record_run(parser, args, output)
    train_loader, val_loader = build_loaders(args, config, sampling, device)
    model = build_model(args.arch, config).to(device)
    if args.compile:
        compile_networks(model)
    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs)
    scaler = torch.amp.GradScaler('cuda', enabled=device.type == 'cuda')
    first, best = 0, float('inf')
    if args.resume:
        first, best = restore_training(output/'last.pt', model, optimizer, scheduler, scaler)
    print(f'device={device} parameters={sum(p.numel() for p in model.parameters()):,} '
          f'train_windows/epoch<={len(train_loader.dataset):,} val_windows<={len(val_loader.dataset):,} '
          f'(events from {config.context_s:.2f} s before the first query)', flush=True)
    if args.max_clips or args.max_train_batches or args.max_val_batches:
        print('DEVELOPMENT RUN: metrics are not full-dataset performance.', flush=True)
    losses = dict(bone_weight=args.bone_weight, calibration_weight=args.calibration_weight,
                  world_weight=args.world_weight if sampling.target_frame == 'rig' else 0.)
    for epoch in range(first, args.epochs):
        train_loader.dataset.epoch = epoch
        train = run_epoch(model, train_loader, device, optimizer, scaler, args.max_train_batches, **losses)
        val = run_epoch(model, val_loader, device, limit=args.max_val_batches, **losses)
        scheduler.step()
        improved = val['mpjpe_mm'] < best
        best = min(best, val['mpjpe_mm'])
        with (output/'metrics.jsonl').open('a', encoding='utf-8') as f:
            f.write(json.dumps(dict(epoch=epoch+1, train=train, val=val))+'\n')
        saved = training_checkpoint(model, sampling, optimizer, scheduler, scaler, epoch, best, options)
        save_checkpoint(output/'last.pt', saved)
        if improved:
            save_checkpoint(output/'best.pt', saved)
        print(f"epoch={epoch+1}/{args.epochs} train={train['mpjpe_mm']:.2f}mm val={val['mpjpe_mm']:.2f}mm "
              f"PCK20={val['pck20']:.3f}", flush=True)


if __name__ == '__main__':
    main()


In [ ]:
%%writefile /content/gigahands_training/training/evaluate.py
"""Evaluate a checkpoint, or the model-free HandLite anchor baseline, on fixed validation windows."""
import argparse
from dataclasses import replace
import json
from pathlib import Path
import torch
from torch.utils.data import DataLoader
from hand_tracking.checkpoints import architecture_of, load_checkpoint
from hand_tracking.config import LiteConfig, SamplingConfig
from hand_tracking.data import HandWindows
from hand_tracking.engine import evaluate_anchor, run_epoch


def parse_args(argv=None):
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument('--split', choices=['val','test'], default='val', help='Use test only after model selection')
    p.add_argument('--checkpoint', help='Trained model; omit with --triangulation-only')
    p.add_argument('--triangulation-only', action='store_true',
                   help='Evaluate the model-free anchor baseline (HandLite geometry, no learned correction)')
    p.add_argument('--data', default='exports/gigahands_balanced_5gb')
    p.add_argument('--output', required=True)
    p.add_argument('--batch-size', type=int, default=16)
    p.add_argument('--windows-per-clip', type=int, default=16, help='0 evaluates all validation windows')
    p.add_argument('--workers', type=int, default=0)
    p.add_argument('--device', choices=['cpu', 'cuda'], default='cpu')
    p.add_argument('--max-clips', type=int, default=0, help='Development subset; 0 uses all clips')
    p.add_argument('--max-events', type=int, help='Override checkpoint input layout; baseline default is 128')
    args = p.parse_args(argv)
    if bool(args.checkpoint) == args.triangulation_only:
        p.error('Give exactly one of --checkpoint or --triangulation-only')
    return args


def main(argv=None):
    args = parse_args(argv)
    torch.set_num_threads(4)
    device = torch.device(args.device)
    if args.triangulation_only:
        # Same window/history layout and geometry as the default HandLite.
        checkpoint = None
        config = LiteConfig()
        sampling = SamplingConfig()
    else:
        checkpoint = load_checkpoint(args.checkpoint, device)
        config, sampling = checkpoint.model.config, checkpoint.sampling
    if args.max_events is not None:
        sampling = replace(sampling, max_events=args.max_events)
    dataset = HandWindows(args.data, args.split, args.windows_per_clip, max_clips=args.max_clips,
                          context_s=config.context_s, max_events=sampling.max_events, target_frame=sampling.target_frame,
                          seen_span_s=config.event_span_s)   # as training: hands the input never shows are not scored
    loader = DataLoader(dataset, batch_size=args.batch_size, num_workers=args.workers)
    if checkpoint is None:
        report = evaluate_anchor(loader, device, config)
        report.update(method='triangulation_anchor')
    else:
        options = checkpoint.saved['training_options']
        report = run_epoch(checkpoint.model, loader, device, bone_weight=options['bone_weight'],
                           calibration_weight=options.get('calibration_weight', 0.),
                           world_weight=options.get('world_weight', 0.) if sampling.target_frame == 'rig' else 0.)
        report.update(method=architecture_of(checkpoint.model), checkpoint=str(Path(args.checkpoint).resolve()))
    report.update(split=args.split, windows_per_clip=args.windows_per_clip, max_events=sampling.max_events,
                  target_frame=sampling.target_frame,
                  evaluated_clips=len(dataset.rows), limited_clips=bool(args.max_clips))
    with Path(args.output).open('x', encoding='utf-8') as f:
        json.dump(report, f, indent=2)
    print(json.dumps(report), flush=True)


if __name__ == '__main__':
    main()


In [ ]:
%%writefile /content/gigahands_training/training/predict.py
"""Predict one NPZ window from a trained checkpoint."""
import argparse
from pathlib import Path
import numpy as np
import torch
from hand_tracking.checkpoints import load_checkpoint
from hand_tracking.contracts import model_inputs
from hand_tracking.data import read_clip, make_sample


def parse_args(argv=None):
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument('--checkpoint', required=True)
    p.add_argument('--input', required=True)
    p.add_argument('--output', required=True)
    p.add_argument('--window-index', type=int, default=-1)
    return p.parse_args(argv)


def main(argv=None):
    args = parse_args(argv)
    torch.set_num_threads(4)
    checkpoint = load_checkpoint(args.checkpoint)
    model = checkpoint.model
    clip = read_clip(args.input)
    window = args.window_index % len(clip['window_starts'])
    sample = make_sample(clip, window, context_s=model.config.context_s, max_events=checkpoint.sampling.max_events)
    if not sample['event_present'].any():
        raise ValueError('No arrived observations in this window')
    with torch.inference_mode():
        output = model(*(value[None] for value in model_inputs(sample)), return_details=True)
    if not bool(output.accepted[0, -1]):
        raise ValueError('The model used no event for the final query of this window')
    pose = output.pose[0, -1].numpy()
    # The camera correction used for the final query, when the model has one.
    extra = {} if output.calibration is None else dict(calibration=output.calibration[0, -1].numpy())
    with Path(args.output).open('xb') as stream:
        np.savez_compressed(stream, predicted_xyz=pose, predicted_xyz_cm=pose*clip['world_unit_cm'],
                            target_xyz=sample['target'][-1].numpy(), target_mask=sample['target_mask'][-1].numpy(),
                            world_unit_cm=clip['world_unit_cm'], window_index=window, **extra)
    print(f'Saved {args.output}: [2,21,3] world coordinates and cm')


if __name__ == '__main__':
    main()


In [ ]:
%%writefile /content/gigahands_training/training/export.py
"""Export a trained checkpoint (HandLite or HandDirect) for deployment (ONNX and/or ncnn) and
check the torch-free runtime against the PyTorch stream on a real clip.
Requires requirements-export.txt (ncnn: also `ncnn` and `pnnx`)."""
import argparse
import json
from pathlib import Path
import numpy as np
import torch
from hand_tracking.checkpoints import architecture_of, load_checkpoint, stream_for
from hand_tracking.data import read_clip
from hand_tracking.direct_export import export_direct
from hand_tracking.direct_runtime import DirectRuntime
from hand_tracking.lite_export import FORMATS, export_lite
from hand_tracking.lite_runtime import LiteRuntime

BACKENDS = {'onnx': 'onnxruntime', 'ncnn': 'ncnn'}
# architecture -> (export function, torch-free runtime)
DEPLOYMENT = {'lite': (export_lite, LiteRuntime), 'direct': (export_direct, DirectRuntime)}


def parse_args(argv=None):
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument('--checkpoint', required=True, help='A trained checkpoint (any architecture)')
    p.add_argument('--output', required=True, help='New directory for the graphs and their metadata')
    p.add_argument('--formats', nargs='+', choices=FORMATS, default=list(FORMATS))
    p.add_argument('--check-input', help='NPZ clip streamed through each runtime and the PyTorch stream')
    p.add_argument('--queries', type=int, default=30, help='Query times compared from the clip (> 0)')
    p.add_argument('--tolerance', type=float, default=2e-3,
                   help='Largest accepted |runtime - PyTorch| in world units (ncnn may use fp16)')
    args = p.parse_args(argv)
    if args.queries < 1 or not args.tolerance > 0:
        p.error('--queries and --tolerance must be positive')
    return args


def compare_on_clip(model, directory, formats, npz, queries):
    """Stream the clip's events through the PyTorch stream and each runtime; largest pose difference."""
    clip = read_clip(npz)
    order = np.argsort(clip['event_arrival'], kind='stable')
    times = clip['query_time'][np.linspace(len(clip['query_time'])//4, len(clip['query_time'])-1, queries).astype(int)]
    runtime = DEPLOYMENT[architecture_of(model)][1]
    runners = {'pytorch': stream_for(model)}
    runners.update({name: runtime(directory, backend=BACKENDS[name]) for name in formats})
    worst, pushed, compared = {name: 0. for name in formats}, 0, 0
    for time in times:
        while pushed < len(order) and clip['event_arrival'][order[pushed]] <= time:
            i = order[pushed]
            pushed += 1
            for runner in runners.values():
                runner.push(clip['event_camera'][i], clip['event_features'][i], clip['event_valid'][i],
                            clip['event_capture'][i], clip['event_arrival'][i])
        if not pushed:
            continue
        expected = runners['pytorch'].query(time).numpy()
        for name in formats:
            worst[name] = max(worst[name], pose_difference(runners[name].query(time), expected, name, time))
        compared += 1
    if not compared:
        raise ValueError(f'No query of {npz} had an arrived event to compare')
    return worst


def pose_difference(actual, expected, name, time):
    """Largest |actual - expected|; non-finite values fail (max() would silently drop NaN)."""
    if not np.isfinite(expected).all():
        raise FloatingPointError(f'PyTorch pose is not finite at {time}')
    if not np.isfinite(actual).all():
        raise FloatingPointError(f'{name} pose is not finite at {time}')
    return float(np.abs(actual-expected).max())


def main(argv=None):
    args = parse_args(argv)
    torch.set_num_threads(4)
    checkpoint = load_checkpoint(args.checkpoint)
    export = DEPLOYMENT[architecture_of(checkpoint.model)][0]
    meta = export(checkpoint.model, args.output, args.formats)
    report = dict(output=str(Path(args.output).resolve()), formats=args.formats, config=meta['config'],
                  graphs=meta['graphs'])
    if args.check_input:
        report['max_abs_difference'] = compare_on_clip(checkpoint.model, args.output, args.formats, args.check_input,
                                                       args.queries)
    print(json.dumps(report, indent=2), flush=True)
    if args.check_input and max(report['max_abs_difference'].values()) > args.tolerance:
        raise SystemExit(f'Runtime differs from PyTorch by more than {args.tolerance}')


if __name__ == '__main__':
    main()


In [ ]:
%%writefile /content/gigahands_training/hand_tracking/__init__.py
"""Event-based multi-camera hand tracking components."""


In [ ]:
%%writefile /content/gigahands_training/hand_tracking/checkpoints.py
"""Checkpoints: model and input layout together (including pre-refactor checkpoints), and
the full training state for resuming. `architecture` names the model class."""
from dataclasses import asdict, dataclass
import torch
from .config import SamplingConfig, saved_config
from .direct import DirectStream, HandDirect
from .lite import HandLite, LiteStream

MODELS = {'lite': HandLite, 'direct': HandDirect}
STREAMS = {HandLite: LiteStream, HandDirect: DirectStream}


def architecture_of(model):
    return next(name for name, model_type in MODELS.items() if isinstance(model, model_type))


def build_model(architecture, config):
    return MODELS[architecture](config)


def stream_for(model):
    """Event-by-event runner of the model (results equal forward())."""
    return STREAMS[type(model)](model)


INPUT_CONTRACT = ('events [B,E,2,21,14]+valid+camera+capture/arrival seconds+present, query '
                  'times [B,Q] seconds -> pose [B,Q,2,21,3], calibration per query [B,Q,3,6] (lite); '
                  'or stream_for(model) event by event; XYZ world units')


@dataclass
class LoadedCheckpoint:
    model: torch.nn.Module
    sampling: SamplingConfig
    saved: dict


def save_checkpoint(path, value):
    """Write atomically: a crash never leaves a truncated checkpoint at path."""
    temporary = path.with_suffix('.tmp')
    torch.save(value, temporary)
    temporary.replace(path)


def load_checkpoint(path, device='cpu'):
    saved = torch.load(path, map_location='cpu', weights_only=True)
    architecture = saved.get('architecture')
    if architecture not in MODELS:
        raise ValueError(f'Unsupported checkpoint architecture {architecture!r} '
                         f'(HandTransformer was removed); choose from {sorted(MODELS)}')
    model = build_model(architecture, saved_config(architecture, saved['model_config'])).to(device)
    model.load_state_dict(saved['model'])
    model.eval()
    return LoadedCheckpoint(model, SamplingConfig.from_checkpoint(saved), saved)


def training_checkpoint(model, sampling, optimizer, scheduler, scaler, epoch, best_val_mpjpe_mm, options):
    """Everything needed to resume after `epoch` (0-based) and to load the model alone."""
    cuda = next(model.parameters()).device.type == 'cuda'
    return dict(model=model.state_dict(), architecture=architecture_of(model), model_config=asdict(model.config),
                sampling_config=asdict(sampling),
                optimizer=optimizer.state_dict(), scheduler=scheduler.state_dict(), scaler=scaler.state_dict(),
                epoch=epoch, best_val_mpjpe_mm=best_val_mpjpe_mm, rng=torch.get_rng_state(),
                cuda_rng=torch.cuda.get_rng_state_all() if cuda else None, training_options=options,
                torch_version=str(torch.__version__), parameters=sum(p.numel() for p in model.parameters()),
                input_contract=INPUT_CONTRACT)


def restore_training(path, model, optimizer, scheduler, scaler):
    """Load a training_checkpoint into existing objects; returns (next epoch, best val MPJPE)."""
    saved = torch.load(path, map_location='cpu', weights_only=True)
    model.load_state_dict(saved['model'])
    optimizer.load_state_dict(saved['optimizer'])
    scheduler.load_state_dict(saved['scheduler'])
    scaler.load_state_dict(saved['scaler'])
    torch.set_rng_state(saved['rng'])
    if next(model.parameters()).device.type == 'cuda' and saved['cuda_rng'] is not None:
        torch.cuda.set_rng_state_all(saved['cuda_rng'])
    return saved['epoch']+1, saved['best_val_mpjpe_mm']


In [ ]:
%%writefile /content/gigahands_training/hand_tracking/config.py
"""Model and input-layout settings. Torch-free, so deployment runtimes can read them. The CLI is generated from a config's fields, so a new field is
automatically a training flag (see add_model_arguments)."""
from dataclasses import dataclass, field, fields
from .constants import NUM_HANDS, FINGERS


# Arrival can trail capture by the pipeline delay; windows carry this much extra history.
ARRIVAL_MARGIN_S = .25


def _option(default, help):
    return field(default=default, metadata=dict(help=help))


@dataclass
class LiteConfig:
    """HandLite: fixed-size, export-friendly model (hand_tracking.lite)."""
    dim: int = _option(64, 'Token width')
    heads: int = _option(4, 'Attention heads')
    blocks: int = _option(2, 'Decoder blocks (cross-attention to hand tokens, then over joints)')
    dropout: float = _option(.1, 'Dropout probability (residual branches)')
    attention_dropout: float = _option(0., 'Dropout probability of attention weights (costly on large maps)')
    # A query uses each camera's slots_per_camera most recent events captured within
    # event_span_s: a fixed-size input whatever the frame rate or number of events.
    slots_per_camera: int = _option(8, 'Most recent events per camera a query uses')
    event_span_s: float = _option(.5, 'Oldest event capture a query uses (s)')
    calibration_head: bool = _option(True, 'Learn a per-query camera calibration from the slots')
    calibration_rotation_deg: float = _option(5., 'Learned calibration rotation scale (degrees)')
    calibration_position: float = _option(.1, 'Learned calibration shift scale (world units)')
    gap_embedding: bool = _option(True, 'Add a learned capture-to-query gap embedding to decoder hand tokens')
    finger_tokens: bool = _option(True, 'Encode each hand as one token per finger (with the wrist), not one token')
    sample_max_age_s: float = _option(.2, 'Oldest ray combined into a triangulation sample (s)')
    anchor_lookback_s: float = _option(.2, 'Anchor velocity fit window (s); 0 holds the last sample')
    anchor_hold_s: float = _option(.3, 'Oldest sample an anchor may hold (s)')
    # A joint seen by fewer than two cameras is anchored on its newest ray, at the depth of
    # its anchor or of its last triangulation within anchor_depth_memory_s.
    anchor_ray_depth: bool = _option(True, 'Anchor joints seen by one camera on that ray at a remembered depth')
    anchor_depth_memory_s: float = _option(1., 'Oldest triangulation whose depth a single-ray anchor keeps (s)')
    # Query-time triangulation of calibrated rays drops one ray of three or more when it is
    # this many times further from the others' point than they are (0: off). Not applied to
    # the arrival-time nominal triangulation: a miscalibrated camera looks like an outlier.
    # Off by default: on clean simulated windows it cost 0.2 mm with the true calibration but
    # 13 mm with none (a calibrator not yet trained); 6 separated hand-swap outliers well.
    ray_outlier_ratio: float = _option(0., 'Drop a ray this many times less consistent than the others (0: off; try 6)')

    def __post_init__(self):
        if (self.dim < 1 or self.heads < 1 or self.dim % self.heads or self.blocks < 1 or not 0 <= self.dropout < 1
                or not 0 <= self.attention_dropout < 1
                or self.slots_per_camera < 1 or self.anchor_lookback_s < 0 or self.ray_outlier_ratio < 0
                or min(self.event_span_s, self.sample_max_age_s, self.anchor_hold_s,
                       self.calibration_rotation_deg, self.calibration_position, self.anchor_depth_memory_s) <= 0):
            raise ValueError('Invalid model configuration')

    @property
    def context_s(self):
        """Seconds of events a window needs before its first query (slots, remembered
        triangulations and their rays)."""
        memory = self.anchor_depth_memory_s if self.anchor_ray_depth else 0.
        return (max(self.event_span_s, self.anchor_hold_s, self.anchor_lookback_s, memory) + ARRIVAL_MARGIN_S
                + self.sample_max_age_s)

    @property
    def anchor_features(self):
        """Decoder input per joint: anchor (3), flags (5) and, with anchor_ray_depth, single-ray (1)."""
        return 8+self.anchor_ray_depth

    @property
    def event_tokens(self):
        """Tokens the encoder makes per event: per hand one per finger, or one."""
        return NUM_HANDS*(FINGERS if self.finger_tokens else 1)


@dataclass
class DirectConfig:
    """HandDirect: networks only, joints straight from the events (hand_tracking.direct)."""
    dim: int = _option(64, 'Token width')
    heads: int = _option(4, 'Attention heads')
    fusion_blocks: int = _option(2, 'Self-attention blocks over the slot tokens (views and times)')
    blocks: int = _option(2, 'Decoder blocks (cross-attention to the tokens, then over joints)')
    dropout: float = _option(.1, 'Dropout probability (residual branches)')
    attention_dropout: float = _option(0., 'Dropout probability of attention weights')
    # A query uses each camera's slots_per_camera most recent events captured within
    # event_span_s: a fixed-size input whatever the frame rate or number of events.
    slots_per_camera: int = _option(8, 'Most recent events per camera a query uses')
    event_span_s: float = _option(.5, 'Oldest event capture a query uses (s)')

    def __post_init__(self):
        if (self.dim < 1 or self.heads < 1 or self.dim % self.heads or self.blocks < 1 or self.fusion_blocks < 0
                or not 0 <= self.dropout < 1 or not 0 <= self.attention_dropout < 1 or self.slots_per_camera < 1
                or self.event_span_s <= 0):
            raise ValueError('Invalid model configuration')

    @property
    def context_s(self):
        """Seconds of events a window needs before its first query."""
        return self.event_span_s+ARRIVAL_MARGIN_S

    @property
    def event_tokens(self):
        """Tokens the encoder makes per event: one per finger of each hand."""
        return NUM_HANDS*FINGERS


ARCHITECTURES = {'lite': LiteConfig, 'direct': DirectConfig}
# Fields added after checkpoints/exports were written: their value for a saved config
# without the field (the model it was built as), whatever the current default.
LEGACY_OPTIONS = {'lite': dict(gap_embedding=False, finger_tokens=False, anchor_ray_depth=False, ray_outlier_ratio=0.)}


def config_class(architecture):
    try:
        return ARCHITECTURES[architecture]
    except KeyError:
        raise ValueError(f'Unknown architecture {architecture!r}; choose from {sorted(ARCHITECTURES)}') from None


def saved_config(architecture, values):
    """Config of a saved checkpoint or export, with LEGACY_OPTIONS for fields it predates."""
    return config_class(architecture)(**{**LEGACY_OPTIONS.get(architecture, {}), **values})


def _flag(name, default):
    """calibration_head (default True) -> --no-calibration-head; others -> --name-with-dashes."""
    if default is True:
        return '--no-'+name.replace('_', '-'), 'no_'+name
    return '--'+name.replace('_', '-'), name


def add_model_arguments(parser, config_type=LiteConfig):
    """One flag per config field, with the field's default and help."""
    group = parser.add_argument_group('model')
    for item in fields(config_type):
        flag, dest = _flag(item.name, item.default)
        if isinstance(item.default, bool):
            text = item.metadata['help']
            group.add_argument(flag, dest=dest, action='store_true', help=f'Disable: {text}' if item.default else text)
        else:
            kind = float if item.default is None else type(item.default)
            group.add_argument(flag, dest=dest, type=kind, default=item.default, help=item.metadata['help'])


def model_config_from_args(args, config_type=LiteConfig):
    values = {}
    for item in fields(config_type):
        _, dest = _flag(item.name, item.default)
        value = getattr(args, dest)
        values[item.name] = (not value) if item.default is True else value
    return config_type(**values)


@dataclass
class SamplingConfig:
    """Input layout settings stored alongside, rather than inside, model weights.
    target_frame: coordinates of the pose targets, 'rig' (the nominal rig as actually placed;
    rig-wide errors no camera reveals removed, see data.rig_frame) or 'world'."""
    max_events: int = 128
    target_frame: str = 'rig'

    def __post_init__(self):
        if self.max_events < 1:
            raise ValueError('max_events must be positive')
        if self.target_frame not in ('rig', 'world'):
            raise ValueError("target_frame must be 'rig' or 'world'")

    @classmethod
    def from_checkpoint(cls, saved):
        if 'sampling_config' in saved:
            # Checkpoints before target_frame were trained on world-frame targets.
            return cls(**{'target_frame': 'world', **saved['sampling_config']})
        # Checkpoints saved before the module refactor stored this in CLI options.
        options = saved.get('training_options', {})
        return cls(max_events=options.get('max_events', cls.max_events))


In [ ]:
%%writefile /content/gigahands_training/hand_tracking/constants.py
"""Input layout, units and tolerances shared by data, geometry, model and stream."""

NUM_CAMERAS = 3
NUM_HANDS = 2
NUM_JOINTS = 21
# One camera frame gives one token per joint of both hands.
HAND_JOINTS = NUM_HANDS*NUM_JOINTS
# Joint order: wrist 0, then four joints per finger (thumb 1-4, ..., little 17-20).
# A finger group is the wrist and that finger's joints.
FINGERS = 5
FINGER_JOINTS = tuple((0,)+tuple(range(1+4*f, 5+4*f)) for f in range(FINGERS))

# Exported per-joint feature channels [...,FEATURE_CHANNELS]. Channels 8-9 (capture and
# arrival relative to the export query) and 11-13 (ids) are not model inputs: the model
# uses the event's own capture/arrival times instead.
FEATURE_CHANNELS = 14
MODEL_CHANNELS = 11
UV = slice(0, 2)               # normalised pixel coordinates in [0,1]
RAY_ORIGIN = slice(2, 5)       # nominal camera centre, world units
RAY_DIRECTION = slice(5, 8)    # nominal ray direction, world axes
DELAY = slice(10, 11)          # capture->arrival delay, seconds

# Learned features express times, ages and delays in this unit so they are O(1).
TIME_UNIT_S = .1
# Ray-to-point miss vectors are scaled to O(1) for the same reason.
MISS_SCALE = 10.
# Per-camera calibration: rotation vector (3) and centre shift (3).
CALIBRATION_PARAMS = 6

# An event is available to a query when it arrived by the query time, up to float32
# rounding of times relative to the batch origin.
ARRIVAL_TOLERANCE_S = 1e-6
# Distinct sample stamps are >=1/camera_fps/3 apart; float32 relative times differ by ~1e-7.
REPEAT_TOLERANCE_S = 1e-5
# Velocity needs samples spread over at least this time std; extrapolation is capped.
MIN_TIME_STD_S = .005
MAX_EXTRAPOLATION_S = .25
# Triangulation needs two rays at least this far from parallel: sin^2 of the widest
# angle between them; 2e-3 is about 2.6 degrees.
MIN_RAY_SIN2 = 2e-3
# A single-ray anchor keeps a reference point's depth along the ray, at least this far
# in front of the camera (world units).
MIN_RAY_DEPTH = 1e-3
# Outlier ray test: a ray is rejected when its angular residual to the point of the other
# rays exceeds the ratio times their own residual, floored at this many radians (about
# 3 px of detector noise) so a perfectly consistent pair does not make every ray an outlier.
RAY_RESIDUAL_FLOOR = .01
# The outlier must stand out: its ratio this many times the next candidate's; otherwise
# two rays are suspects and the sample is dropped as ambiguous.
OUTLIER_MARGIN = 3.
# Additive attention bias of an excluded key; finite so fp16 runtimes stay NaN-free.
MASK_OFF = -1e4
# Streams keep events this long beyond the model's context before pruning.
STREAM_KEEP_MARGIN_S = .5


def check_event(camera, features_shape, valid_shape):
    """Validate one camera frame for a stream; returns the camera index."""
    if int(camera) != camera or not 0 <= int(camera) < NUM_CAMERAS:
        raise ValueError(f'camera must be an integer in [0, {NUM_CAMERAS}), got {camera!r}')
    if tuple(features_shape) != (NUM_HANDS, NUM_JOINTS, FEATURE_CHANNELS) or tuple(valid_shape) != (NUM_HANDS, NUM_JOINTS):
        raise ValueError(f'Expected features [2,21,14] and valid [2,21], got {tuple(features_shape)} and {tuple(valid_shape)}')
    return int(camera)


In [ ]:
%%writefile /content/gigahands_training/hand_tracking/contracts.py
"""Tensor contracts shared by loaders, model, and streaming inference.

Model times are float32 seconds relative to a nearby origin. Stream storage keeps
absolute timestamps in float64. Shapes below include the batch dimension.
"""
from typing import NamedTuple, TypedDict
from torch import Tensor

MODEL_INPUT_KEYS = ('event_features', 'event_valid', 'event_camera', 'event_capture',
                    'event_arrival', 'event_present', 'query_times')


# Supervision and unit conversion; never model inputs.
TARGET_KEYS = ('target', 'target_world', 'target_mask', 'calibration_target', 'world_unit_cm')


def model_inputs(batch):
    return tuple(batch[key] for key in MODEL_INPUT_KEYS)


class WindowSample(TypedDict):
    """One training/evaluation window (data.make_sample); a DataLoader adds a batch dim.
    Times are float32 seconds relative to the window's last query."""
    event_features: Tensor      # [E,2,21,14], invalid joints zeroed
    event_valid: Tensor         # [E,2,21]
    event_camera: Tensor        # [E] long
    event_present: Tensor       # [E], False = padding
    event_capture: Tensor       # [E]
    event_arrival: Tensor       # [E]
    query_times: Tensor         # [Q]
    target: Tensor              # [Q,2,21,3] world units, in the sampling target_frame (rig or world)
    target_world: Tensor        # [Q,2,21,3] the same in the simulator's true world frame
    target_mask: Tensor         # [Q,2,21]
    calibration_target: Tensor  # [3,6] in the target frame, NaN when unknown (real captures)
    world_unit_cm: Tensor       # []


WINDOW_SAMPLE_KEYS = tuple(WindowSample.__annotations__)
assert set(WINDOW_SAMPLE_KEYS) == set(MODEL_INPUT_KEYS+TARGET_KEYS)


class EventBatch(TypedDict):
    raw: Tensor       # [B,E,2,21,MODEL_CHANNELS], invalid joints zeroed
    valid: Tensor     # [B,E,2,21]
    camera: Tensor    # [B,E]
    capture: Tensor   # [B,E], relative seconds
    arrival: Tensor   # [B,E], relative seconds
    present: Tensor   # [B,E], excludes padding and stale captures


class RawEvents(TypedDict):
    features: Tensor
    valid: Tensor
    camera: Tensor
    capture: Tensor   # stream: [E], absolute float64 seconds
    arrival: Tensor


class ModelOutput(NamedTuple):
    """model(..., return_details=True): calibration per query [B,Q,3,6] (None for a model
    without camera correction); accepted marks the queries with at least one slot."""
    pose: Tensor         # [B,Q,2,21,3] world units
    calibration: Tensor  # [B,Q,3,6] camera correction (see geometry.apply_calibration)
    accepted: Tensor     # [B,Q]


In [ ]:
%%writefile /content/gigahands_training/hand_tracking/data.py
"""Event windows for the time-based model, read from simulator/GigaHands NPZ exports.
Clip-local streaming avoids repeatedly decompressing the same NPZ per window. Samples
follow contracts.WindowSample."""
import json
import os
from pathlib import Path
import numpy as np
import torch
from torch.utils.data import IterableDataset, get_worker_info
from .config import LiteConfig, SamplingConfig
from .constants import NUM_CAMERAS, NUM_HANDS, NUM_JOINTS, FEATURE_CHANNELS, CALIBRATION_PARAMS, RAY_ORIGIN, \
    RAY_DIRECTION
from .contracts import WindowSample

SPLITS = ('train', 'val', 'test')
# NPZ arrays a clip keeps: model inputs, targets and window indexing.
CLIP_ARRAYS = ('features', 'input_mask', 'target_xyz', 'target_mask', 'query_time', 'window_starts')
# Frame identity and model-visible (reported) timing, used only to recover events.
TIMING_ARRAYS = ('selected_frame', 'capture_time', 'arrival_time')
CALIBRATION_ARRAYS = ('actual_rotations', 'nominal_rotations', 'actual_origins', 'nominal_origins')
# Target frames: 'rig' removes each clip's rig-wide error, which no camera observation
# reveals; 'world' keeps the simulator's true world coordinates.
TARGET_FRAMES = ('rig', 'world')
# Prior scale of the per-camera corrections the rig frame keeps small (the calibrator's).
RIG_FRAME_ROTATION_STD_DEG = 5.
RIG_FRAME_POSITION_STD = .1
# read_clip's arrays that make_sample uses; the cache (cached_clip) keeps only these.
SAMPLE_ARRAYS = ('event_features', 'event_valid', 'event_camera', 'event_capture', 'event_arrival', 'target_xyz',
                 'target_mask', 'query_time', 'window_starts', 'calibration_target', 'rig_target',
                 'rig_calibration_target')
CACHE_VERSION = 1   # bump when read_clip's output changes


def read_manifest(root):
    """manifest.jsonl rows, checked for valid paths and no participant/source train-val leakage."""
    root = Path(root).resolve()
    lines = (root/'manifest.jsonl').read_text(encoding='utf-8').splitlines()
    rows = [json.loads(line) for line in lines if line.strip()]
    people = {split: set() for split in SPLITS}
    sources = {split: set() for split in SPLITS}
    files = set()
    for row in rows:
        split = row.get('split')
        if split not in people:
            raise ValueError('Manifest requires train/val/test records without errors')
        path = (root/row['file']).resolve()
        if not path.is_relative_to(root) or not path.is_file():
            raise ValueError('Invalid dataset path')
        if row['file'] in files:
            raise ValueError('Duplicate output path')
        files.add(row['file'])
        people[split].add(row['participant'])
        sources[split].add(row['source'])
    if not people['train'] or not people['val']:
        raise ValueError('Both train and val must be present')
    for i, first in enumerate(SPLITS):
        for second in SPLITS[i+1:]:
            if people[first] & people[second] or sources[first] & sources[second]:
                raise ValueError('Participant or source leakage between splits')
    return rows


def rotation_log(matrix):
    """Rotation matrix -> rotation vector (angles well below 180 degrees)."""
    angle = np.arccos(np.clip((np.trace(matrix)-1)/2, -1, 1))
    axis = np.array([matrix[2,1]-matrix[1,2], matrix[0,2]-matrix[2,0], matrix[1,0]-matrix[0,1]])
    return axis*(angle/(2*np.sin(angle)) if angle > 1e-8 else .5)


def rotation_exp(vector):
    """Rotation vector [3] -> matrix (Rodrigues)."""
    angle = np.linalg.norm(vector)
    if angle < 1e-12:
        return np.eye(3)
    k = np.cross(np.eye(3), vector/angle)
    return np.eye(3)+np.sin(angle)*k+(1-np.cos(angle))*k@k


def rig_frame(data, steps=10, tolerance=1e-10):
    """Similarity (rotation [3,3], shift [3], scale) from the true world to the clip's rig
    frame, or None without the simulator rig (real captures).

    Rays only reveal the cameras relative to each other: rotating, moving or scaling the
    whole rig with the hands changes no observation. The rig frame is the true world moved
    by the similarity that leaves the smallest per-camera corrections from the nominal rig
    (rotation / RIG_FRAME_ROTATION_STD_DEG and shift / RIG_FRAME_POSITION_STD, least
    squares, Gauss-Newton): the nominal rig as it was actually placed. Hands keep their
    position around the nominal rig's origin; only the unobservable rig-wide error goes.
    """
    if not all(key in data for key in CALIBRATION_ARRAYS):
        return None
    actual, nominal, actual_origin, nominal_origin = (np.asarray(data[key], np.float64) for key in CALIBRATION_ARRAYS)
    corrections = [a.T@n for a, n in zip(actual, nominal)]                  # calibration_target's rotations
    scales = np.r_[[np.deg2rad(RIG_FRAME_ROTATION_STD_DEG)]*3, [RIG_FRAME_POSITION_STD]*3]

    def residuals(x):
        rotation, shift, scale = rotation_exp(x[:3]), x[3:6], np.exp(x[6])
        rows = [np.r_[rotation_log(rotation@c), scale*rotation@o+shift-n]/scales
                for c, o, n in zip(corrections, actual_origin, nominal_origin)]
        return np.concatenate(rows)

    x = np.zeros(7)
    for _ in range(steps):
        r = residuals(x)
        jacobian = np.stack([(residuals(x+e)-residuals(x-e))/2e-6 for e in np.eye(7)*1e-6], -1)
        step = np.linalg.lstsq(jacobian, r, rcond=None)[0]
        x = x-step
        if np.abs(step).max() < tolerance:
            break
    return rotation_exp(x[:3]), x[3:6], float(np.exp(x[6]))


def calibration_target(data):
    """Simulator's true correction of the nominal rays [3,6], NaN when unknown (real captures).

    Same convention as geometry.apply_calibration: world-axis rotation about the camera
    centre (R_actual^T R_nominal, since rays are pixel @ R) and centre shift.
    Supervision only; never a model input.
    """
    if not all(key in data for key in CALIBRATION_ARRAYS):
        return np.full((NUM_CAMERAS, CALIBRATION_PARAMS), np.nan, np.float32)
    rows = [np.concatenate([rotation_log(actual.T@nominal), actual_origin-nominal_origin])
            for actual, nominal, actual_origin, nominal_origin in zip(*(data[key] for key in CALIBRATION_ARRAYS))]
    return np.asarray(rows, np.float32)


def to_rig_frame(target, calibration, data, frame):
    """target [...,3] and calibration target [3,6] in the rig frame (see rig_frame): the hands
    moved by the similarity, the corrections those that turn the nominal rays into the moved
    true rays. Unchanged for real captures, which have no simulator rig."""
    if frame is None:
        return target, calibration
    rotation, shift, scale = frame
    nominal_origin = np.asarray(data['nominal_origins'], np.float64)
    actual_origin = np.asarray(data['actual_origins'], np.float64)
    moved = np.asarray([np.r_[rotation_log(rotation@rotation_exp(row[:3])), scale*rotation@origin+shift-nominal]
                        for row, origin, nominal in zip(calibration.astype(np.float64), actual_origin, nominal_origin)])
    return (scale*target@rotation.T+shift).astype(np.float32), moved.astype(np.float32)


def read_clip(path):
    with np.load(path, allow_pickle=False) as data:
        clip = {key: data[key].copy() for key in CLIP_ARRAYS}
        clip['calibration_target'] = calibration_target(data)
        frame = rig_frame(data)
        clip['rig_target'], clip['rig_calibration_target'] = to_rig_frame(clip['target_xyz'], clip['calibration_target'],
                                                                           data, frame)
        timing = {key: data[key].copy() for key in TIMING_ARRAYS}
        meta = json.loads(str(data['metadata']))
    clip['window'] = int(meta['config']['window'])
    clip['world_unit_cm'] = float(meta['units']['world_unit_cm'])
    clip.update(clip_events(clip, timing))
    return clip


def cached_clip(root, file, cache=None):
    """make_sample's arrays of read_clip(root/file). With a cache folder, the first read also
    writes them uncompressed to cache/file and later reads load that instead, skipping the
    NPZ decompression and the rig-frame fit; an entry is rebuilt when its source changes."""
    path = Path(root)/file
    if cache:
        entry = Path(cache)/file
        source = path.stat()
        stamp = np.array([CACHE_VERSION, source.st_size, source.st_mtime_ns])
        try:
            with np.load(entry, allow_pickle=False) as data:
                if np.array_equal(data['stamp'], stamp):
                    clip = {key: data[key] for key in data.files if key != 'stamp'}
                    clip['window'], clip['world_unit_cm'] = int(clip['window']), float(clip['world_unit_cm'])
                    return clip
        except (OSError, KeyError, ValueError):   # missing, stale format or a partial write
            pass
    clip = read_clip(path)
    clip = {key: clip[key] for key in SAMPLE_ARRAYS + ('window', 'world_unit_cm') if key in clip}
    if cache:
        entry.parent.mkdir(parents=True, exist_ok=True)
        partial = entry.with_name(f'{entry.name}.{os.getpid()}.tmp')
        with partial.open('wb') as f:
            np.savez(f, stamp=stamp, **clip)
        os.replace(partial, entry)   # readers never see a half-written entry
    return clip


def clip_events(clip, timing):
    """Camera frames (events) recovered from the query-grid export, sorted by arrival.

    Each query holds every camera's selected (latest captured, arrived) frame; a frame is
    an event at the first query that selects it, including frames in which no joint was
    detected (they still replace that camera's older rays, as live). Stale out-of-order
    arrivals never get selected, as a stream drops them. A frame superseded before any
    query (two arrivals of one camera between queries) is not recoverable from this format.
    """
    selected = timing['selected_frame']
    new = selected >= 0
    new[1:] &= selected[1:] != selected[:-1]
    frame, camera = np.nonzero(new)
    arrival = timing['arrival_time'][frame, camera].astype(np.float64)
    capture = timing['capture_time'][frame, camera].astype(np.float64)
    # Ties keep (query, camera) order; the model treats input order as arrival order.
    order = np.argsort(arrival, kind='stable')
    return dict(event_features=clip['features'][frame, camera][order],
                event_valid=clip['input_mask'][frame, camera][order],
                event_camera=camera[order].astype(np.int64), event_capture=capture[order],
                event_arrival=arrival[order])


def make_sample(clip, window_index, context_s=None, max_events=SamplingConfig.max_events,
                target_frame=SamplingConfig.target_frame, queries=0, seen_span_s=None) -> WindowSample:
    """Time-based window: the events arrived by the window's last query and captured at most
    context_s before its first query, and the window's query times with their targets.

    Times are seconds relative to the last query. Events are padded to max_events
    (event_present False); if more arrive, the oldest are dropped. target is in target_frame ('rig' or 'world', see
    rig_frame; a clip without the simulator rig has no rig-wide error to remove);
    target_world always in the world frame. queries > 0 keeps only the window's last
    queries (the evaluated last query included) and the events their history needs.
    With seen_span_s (the model's event_span_s), a hand no event captured within it before a
    query (and arrived by it) detected has no target there: nothing in the input shows it.
    """
    if target_frame not in TARGET_FRAMES:
        raise ValueError(f'target_frame must be one of {TARGET_FRAMES}')
    if context_s is None:
        context_s = LiteConfig().context_s
    end = int(clip['window_starts'][window_index])+clip['window']
    if queries < 0 or queries > clip['window']:
        raise ValueError(f"queries must be in [0, {clip['window']}]")
    start = end-queries if queries else end-clip['window']
    query = clip['query_time'][start:end].astype(np.float64)
    last = query[-1]
    arrival, capture = clip['event_arrival'], clip['event_capture']
    chosen = np.nonzero((arrival <= last+1e-9) & (capture >= query[0]-context_s))[0][-max_events:]
    n = len(chosen)
    features = np.zeros((max_events, NUM_HANDS, NUM_JOINTS, FEATURE_CHANNELS), np.float32)
    valid = np.zeros((max_events, NUM_HANDS, NUM_JOINTS), bool)
    camera = np.zeros(max_events, np.int64)
    present = np.zeros(max_events, bool)
    event_capture = np.zeros(max_events)
    event_arrival = np.zeros(max_events)
    features[:n] = clip['event_features'][chosen]
    valid[:n] = clip['event_valid'][chosen]
    camera[:n] = clip['event_camera'][chosen]
    present[:n] = True
    event_capture[:n] = capture[chosen]-last
    event_arrival[:n] = arrival[chosen]-last
    target_mask = clip['target_mask'][start:end].copy()
    if seen_span_s is not None:
        times = (query-last)[:, None]
        usable = (event_arrival[None, :n] <= times+1e-9) & (event_capture[None, :n] >= times-seen_span_s)
        seen = (usable[..., None] & valid[None, :n].any(-1)).any(1)          # [queries, hands]
        target_mask &= seen[..., None]
    world = clip['target_xyz'][start:end].copy()
    if target_frame == 'rig' and 'rig_target' in clip:
        target, calibration = clip['rig_target'][start:end].copy(), clip['rig_calibration_target'].copy()
    else:
        target, calibration = world.copy(), clip['calibration_target'].copy()
    features[~valid] = 0
    if np.any(event_arrival[present] > 1e-6):
        raise ValueError('Future arrival in historical input')
    as_float = lambda value: torch.from_numpy(value.astype(np.float32))
    return WindowSample(event_features=torch.from_numpy(features), event_valid=torch.from_numpy(valid),
                        event_camera=torch.from_numpy(camera), event_present=torch.from_numpy(present),
                        event_capture=as_float(event_capture), event_arrival=as_float(event_arrival),
                        query_times=as_float(query-last), target=as_float(target), target_world=as_float(world),
                        target_mask=torch.from_numpy(target_mask),
                        calibration_target=as_float(calibration),
                        world_unit_cm=torch.tensor(clip['world_unit_cm'], dtype=torch.float32))


def trim_padding(batch):
    """Batch with the event axis cut to its largest present count. make_sample puts padding
    after the events and the model ignores it, so outputs are unchanged; the event axis is
    then as long as the batch needs (max_events is a cap, not a cost)."""
    present = batch['event_present']
    if present.ndim != 2:
        return batch
    used = max(int(present.sum(1).max()), 1)
    if used == present.shape[1] or present[:, used:].any():
        return batch
    return {key: value[:, :used] if key.startswith('event_') else value for key, value in batch.items()}


class HandWindows(IterableDataset):
    """Windows of one split. Train: shuffled clips and random windows, reseeded per epoch
    (set .epoch). Val: fixed, evenly spaced windows. No augmentation: the camera rig is
    fixed, so its world coordinates are worth learning."""

    def __init__(self, root, split, windows_per_clip=16, seed=42, max_clips=0, context_s=None, max_events=SamplingConfig.max_events, target_frame=SamplingConfig.target_frame,
                 queries=0, cache=None, seen_span_s=None):
        super().__init__()
        if context_s is None:
            context_s = LiteConfig().context_s
        if (split not in SPLITS or windows_per_clip < 0 or max_clips < 0 or context_s < 0 or max_events < 1
                or target_frame not in TARGET_FRAMES or queries < 0):
            raise ValueError('Invalid sampling settings')
        self.root = Path(root)
        self.rows = [row for row in read_manifest(root) if row['split'] == split and row['windows'] > 0]
        if max_clips:
            self.rows = self.rows[:max_clips]
        if not self.rows:
            raise ValueError('No usable clips')
        self.split, self.windows_per_clip, self.seed, self.epoch = split, windows_per_clip, seed, 0
        self.context_s, self.max_events, self.target_frame, self.queries = context_s, max_events, target_frame, queries
        self.cache = cache   # folder for cached_clip; None reads the NPZ every time
        self.seen_span_s = seen_span_s   # see make_sample; None keeps every labelled hand

    def _windows(self, count):
        return min(count, self.windows_per_clip) if self.windows_per_clip else count

    def __len__(self):
        return sum(self._windows(row['windows']) for row in self.rows)

    def __iter__(self):
        training = self.split == 'train'
        indices = np.arange(len(self.rows))
        if training:
            np.random.default_rng(self.seed+self.epoch).shuffle(indices)
        worker = get_worker_info()
        if worker:
            indices = indices[worker.id::worker.num_workers]
        for index in indices:
            row = self.rows[index]
            clip = cached_clip(self.root, row['file'], self.cache)
            count = len(clip['window_starts'])
            if count != row['windows']:
                raise ValueError('Manifest window count differs from NPZ')
            if training:
                rng = np.random.default_rng(np.random.SeedSequence([self.seed, self.epoch, int(index)]))
                chosen = rng.choice(count, self._windows(count), replace=False)
            else:
                chosen = np.linspace(0, count-1, self._windows(count), dtype=int)
            for window in chosen:
                sample = make_sample(clip, window, self.context_s, self.max_events, self.target_frame,
                                     self.queries, self.seen_span_s)
                # Event-less windows cannot constrain an observation-based estimator;
                # metrics are reported on the final query, so it needs a target.
                if not sample['event_present'].any() or not sample['target_mask'][-1].any():
                    continue
                yield sample


In [ ]:
%%writefile /content/gigahands_training/hand_tracking/direct.py
"""HandDirect: camera events to the 3D joints of both hands with networks only.

  per event, once, on arrival (from that frame alone)
    features  u,v, ray origin and direction, capture->arrival delay, detected   [2,21,10]
    encoder   one token per finger of each hand                                [10, dim]
  per query at time t
    slots     each camera's slots_per_camera latest events captured within event_span_s
    fusion    self-attention over the slots' tokens, each told its age (t - capture)
    decoder   42 joint queries attend to the fused tokens -> xyz                  [42, 3]

No triangulation, anchor or camera correction: the networks learn how views and times
combine. forward() computes exactly what DirectStream computes event by event.
"""
from dataclasses import dataclass
import torch
from torch import nn
from .config import DirectConfig
from .constants import (NUM_CAMERAS, NUM_HANDS, NUM_JOINTS, HAND_JOINTS, FEATURE_CHANNELS, UV, RAY_ORIGIN,
                        RAY_DIRECTION, DELAY, TIME_UNIT_S, MASK_OFF)
from .contracts import ModelOutput
from .events import _gather, make_events, relative_events, select_slots
from .layers import Block, EventEncoder, key_bias
from .stream import EventStream

# Per joint: u,v (2), ray origin (3) and direction (3), delay (1), detected (1).
DIRECT_JOINT_FEATURES = 10


def direct_features(raw, valid):
    """Event features [...,2,21,11] (invalid joints zeroed), valid -> encoder input [...,2,21,10]."""
    x = torch.cat((raw[..., UV]*2-1, raw[..., RAY_ORIGIN], raw[..., RAY_DIRECTION], raw[..., DELAY]/TIME_UNIT_S,
                   valid.float()[..., None]), -1)
    return torch.where(valid[..., None], x, 0.)


class Fusion(nn.Module):
    """tokens [N,T,dim], validity [N,T] (1/0), ages [N,T] (TIME_UNIT_S) -> fused tokens [N,T,dim].
    Each token first adds an embedding of its age; padding tokens stay zero and are masked."""

    def __init__(self, dim, heads, blocks, dropout, attention_dropout):
        super().__init__()
        self.age = nn.Sequential(nn.Linear(1, 16), nn.GELU(), nn.Linear(16, dim))
        self.blocks = nn.ModuleList([Block(dim, heads, dropout, cross=False, attention_dropout=attention_dropout)
                                     for _ in range(blocks)])

    def forward(self, tokens, valid, ages):
        x = (tokens+self.age(ages[..., None]))*valid[..., None]
        bias = ((valid-1.)*-MASK_OFF)[:, None, None]                                         # [N,1,1,T]
        for block in self.blocks:
            x = block(x, self_bias=bias)*valid[..., None]
        return x


class Decoder(nn.Module):
    """fused tokens [N,T,dim], validity [N,T] (1/0) -> joints [N,42,3]: learned joint queries
    attend to the valid tokens, then to each other."""

    def __init__(self, dim, heads, blocks, dropout, attention_dropout):
        super().__init__()
        self.joints = nn.Parameter(torch.randn(HAND_JOINTS, dim)*.02)
        self.null = nn.Parameter(torch.zeros(1, dim))
        self.blocks = nn.ModuleList([Block(dim, heads, dropout, attention_dropout=attention_dropout)
                                     for _ in range(blocks)])
        self.output = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, dim), nn.GELU(), nn.Linear(dim, 3))

    def forward(self, tokens, valid):
        n, _, d = tokens.shape
        keys = torch.cat((self.null.expand(n, 1, d), tokens), 1)                           # a query with no slot
        bias = torch.cat((torch.zeros_like(valid[:, :1]), (valid-1.)*-MASK_OFF), -1)[:, None, None]
        x = self.joints.expand(n, -1, -1)
        for block in self.blocks:
            x = block(x, keys, bias)
        return self.output(x)


class QueryNetwork(nn.Module):
    """The per-query networks as one graph: fusion then decoder (see lite export)."""

    def __init__(self, fusion, decoder):
        super().__init__()
        self.fusion, self.decoder = fusion, decoder

    def forward(self, tokens, valid, ages):
        return self.decoder(self.fusion(tokens, valid, ages), valid)


@dataclass
class DirectEncoded:
    """Per-event tokens [B,E,10,dim]; they never change after arrival."""
    tokens: torch.Tensor

    def map(self, fn):
        return DirectEncoded(fn(self.tokens))

    def append(self, other, dim=1):
        return DirectEncoded(torch.cat((self.tokens, other.tokens), dim))


class HandDirect(nn.Module):
    def __init__(self, config=None):
        super().__init__()
        self.config = config or DirectConfig()
        c = self.config
        self.encoder = EventEncoder(c.dim, True, DIRECT_JOINT_FEATURES)
        self.query_network = QueryNetwork(Fusion(c.dim, c.heads, c.fusion_blocks, c.dropout, c.attention_dropout),
                                          Decoder(c.dim, c.heads, c.blocks, c.dropout, c.attention_dropout))

    @property
    def slots(self):
        return NUM_CAMERAS*self.config.slots_per_camera

    def encode_events(self, events, targets=None):
        """DirectEncoded of the target events (default: all); each from its own frame only."""
        b, e = events['camera'].shape
        if targets is None:
            targets = torch.arange(e, device=events['camera'].device)
        t = targets.numel()
        features = direct_features(events['raw'][:, targets], events['valid'][:, targets])
        camera = nn.functional.one_hot(events['camera'][:, targets], NUM_CAMERAS).float()
        tokens = self.encoder(features.reshape(b*t, NUM_HANDS, -1), camera.reshape(b*t, NUM_CAMERAS))
        return DirectEncoded(tokens.reshape(b, t, *tokens.shape[1:]))

    def decode_queries(self, events, encoded, query):
        """Pose [B,Q,2,21,3], None (no camera correction) and whether a query had any slot [B,Q]."""
        b, q = query.shape
        n, s, per = b*q, self.slots, self.config.event_tokens
        slot, valid = select_slots(events, query, self.config.slots_per_camera, self.config.event_span_s)
        take = lambda value: _gather(value, slot).reshape(n, s, *value.shape[2:])
        valid = valid.reshape(n, s)
        token_valid = valid.repeat_interleave(per, -1)
        tokens = torch.where(token_valid[..., None], take(encoded.tokens).reshape(n, per*s, -1), 0.)
        ages = torch.where(valid, ((query.reshape(n, 1)-take(events['capture']))/TIME_UNIT_S).clamp_min(0.), 0.)
        pose = self.query_network(tokens, token_valid.to(tokens.dtype), ages.repeat_interleave(per, -1))
        return pose.float().reshape(b, q, NUM_HANDS, NUM_JOINTS, 3), None, valid.any(-1).reshape(b, q)

    def forward(self, event_features, event_valid, event_camera, event_capture, event_arrival, event_present,
                query_times, return_details=False):
        """Model inputs (contracts.MODEL_INPUT_KEYS). return_details gives ModelOutput with no
        calibration and accepted marking queries with any slot [B,Q]."""
        if event_features.ndim != 5 or event_features.shape[2:] != (NUM_HANDS, NUM_JOINTS, FEATURE_CHANNELS):
            raise ValueError('Expected event features [B,E,2,21,14]')
        if event_valid.shape != event_features.shape[:-1] or query_times.ndim != 2:
            raise ValueError('Invalid event/query dimensions')
        events = make_events(event_features, event_valid, event_camera, event_capture, event_arrival, event_present)
        pose, calibration, rows = self.decode_queries(events, self.encode_events(events), query_times.float())
        return ModelOutput(pose, calibration, rows) if return_details else pose


class DirectStream(EventStream):
    """HandDirect event by event: each event's tokens are computed once; a query selects the
    slots and runs fusion and decoder. Equals forward()."""

    def decode(self, time):
        encoded = self.store.encoded.map(lambda value: value[None])
        query = torch.zeros(1, 1, device=self.device)
        return self.model.decode_queries(relative_events(self.store.raw, time), encoded, query)[0][0, 0]


In [ ]:
%%writefile /content/gigahands_training/hand_tracking/direct_export.py
"""Export HandDirect's two fixed-shape networks for direct_runtime: ONNX and ncnn.

    encoder  features [2, 21*10], camera one-hot [3]                     -> tokens [10, dim]
    query    tokens [10*3K, dim], token validity 0/1 [10*3K], ages [10*3K] -> joints [42, 3]
"""
from dataclasses import asdict
from pathlib import Path
import torch
from .constants import NUM_CAMERAS, NUM_HANDS, NUM_JOINTS
from .direct import DIRECT_JOINT_FEATURES
from .direct_runtime import EXPORT_FORMAT, GRAPHS, META_FILE
from .lite_export import FORMATS, write_graphs


def graphs(model):
    """name -> (module, example inputs with a batch of one)."""
    model = model.eval()
    tokens = model.config.event_tokens*model.slots
    torch.manual_seed(0)
    return dict(encoder=(model.encoder, (torch.randn(1, NUM_HANDS, NUM_JOINTS*DIRECT_JOINT_FEATURES),
                                         torch.eye(NUM_CAMERAS)[:1])),
                query=(model.query_network, (torch.randn(1, tokens, model.config.dim), torch.ones(1, tokens),
                                             torch.rand(1, tokens))))


def export_direct(model, directory, formats=FORMATS):
    """Write the graphs and direct.json (config) to a new directory; returns the metadata."""
    unknown = set(formats)-set(FORMATS)
    if unknown:
        raise ValueError(f'Unknown formats {sorted(unknown)}')
    directory = Path(directory)
    directory.mkdir(parents=True)
    return write_graphs(graphs(model), directory, formats, GRAPHS, META_FILE,
                        dict(architecture='direct', export_format=EXPORT_FORMAT, config=asdict(model.config)))


In [ ]:
%%writefile /content/gigahands_training/hand_tracking/direct_runtime.py
"""HandDirect deployment runtime without PyTorch: slot selection around the two exported
networks, run by ncnn or ONNX Runtime. Mirrors hand_tracking.direct.DirectStream.

    runtime = DirectRuntime('exports/direct_model', backend='ncnn')
    runtime.push(camera, features [2,21,14], valid [2,21], capture_time, arrival_time)
    pose = runtime.query(time)          # [2,21,3] world units

Times are absolute seconds (float64). Needs numpy plus `ncnn` or `onnxruntime`.
"""
from dataclasses import dataclass
import json
from pathlib import Path
import numpy as np
from .config import DirectConfig
from .constants import (NUM_CAMERAS, NUM_HANDS, NUM_JOINTS, MODEL_CHANNELS, UV, RAY_ORIGIN, RAY_DIRECTION, DELAY,
                        TIME_UNIT_S, STREAM_KEEP_MARGIN_S, check_event)
from .lite_runtime import NcnnGraphs, OnnxGraphs, select_slot_events

META_FILE = 'direct.json'
EXPORT_FORMAT = 1
GRAPHS = ('encoder', 'query')


def direct_features(raw, valid):
    """Event features [2,21,11] (invalid joints zeroed), valid -> encoder input [2,21,10]."""
    x = np.concatenate((raw[..., UV]*2-1, raw[..., RAY_ORIGIN], raw[..., RAY_DIRECTION], raw[..., DELAY]/TIME_UNIT_S,
                        valid[..., None]), -1)
    return np.where(valid[..., None], x, 0.)


@dataclass
class _Event:
    camera: int
    capture: float
    arrival: float
    tokens: np.ndarray     # [10,dim]


class DirectRuntime:
    def __init__(self, directory, backend='ncnn', threads=1, fp16=True, graphs=None):
        """graphs: any object with run(name, *arrays), instead of loading a backend."""
        directory = Path(directory)
        meta = json.loads((directory/META_FILE).read_text(encoding='utf-8'))
        if meta.get('architecture') != 'direct' or meta.get('export_format') != EXPORT_FORMAT:
            raise ValueError(f'{directory} is not a HandDirect export of format {EXPORT_FORMAT}')
        self.config = DirectConfig(**meta['config'])
        if graphs is not None:
            self.graphs = graphs
        elif backend == 'ncnn':
            self.graphs = NcnnGraphs(directory, threads, fp16, GRAPHS)
        elif backend == 'onnxruntime':
            self.graphs = OnnxGraphs(directory, threads, GRAPHS)
        else:
            raise ValueError("backend must be 'ncnn' or 'onnxruntime'")
        self.keep_s = self.config.context_s+STREAM_KEEP_MARGIN_S
        self.reset()

    def reset(self):
        self.events = []
        self.newest_capture = [None]*NUM_CAMERAS
        self.last_arrival = None

    def __len__(self):
        return len(self.events)

    def push(self, camera, features, valid, capture_time, arrival_time):
        """Encode one arrived camera frame; False (ignored) for a capture no newer than that
        camera's latest. Frames must arrive in order; frames without detections count."""
        camera = check_event(camera, np.shape(features), np.shape(valid))
        capture, arrival = float(capture_time), float(arrival_time)
        if self.last_arrival is not None and arrival < self.last_arrival:
            raise ValueError('Events must be pushed in arrival order')
        self.last_arrival = arrival
        if self.newest_capture[camera] is not None and capture <= self.newest_capture[camera]:
            return False
        self.newest_capture[camera] = capture
        valid = np.array(valid, dtype=bool, copy=True)
        raw = np.where(valid[..., None], np.asarray(features, np.float64)[..., :MODEL_CHANNELS], 0.)
        tokens = self.graphs.run('encoder', direct_features(raw, valid).reshape(NUM_HANDS, -1), np.eye(NUM_CAMERAS)[camera])
        self.events.append(_Event(camera, capture, arrival, np.asarray(tokens)))
        while self.events and self.events[0].capture < arrival-self.keep_s:
            self.events.pop(0)
        return True

    def query(self, time):
        """Pose [2,21,3] at time (not before the latest arrival)."""
        time = float(time)
        if self.last_arrival is None:
            raise ValueError('No events yet')
        if time < self.last_arrival:
            raise ValueError('Query before the latest arrival')
        slots = select_slot_events(self.events, time, self.config.slots_per_camera, self.config.event_span_s)
        per, dim = self.config.event_tokens, self.config.dim
        present = np.array([event is not None for event in slots])
        tokens = np.stack([event.tokens if event is not None else np.zeros((per, dim)) for event in slots])
        ages = np.array([max(time-event.capture, 0.)/TIME_UNIT_S if event is not None else 0. for event in slots])
        joints = self.graphs.run('query', tokens.reshape(-1, dim), np.repeat(present, per), np.repeat(ages, per))
        return np.asarray(joints).reshape(NUM_HANDS, NUM_JOINTS, 3)


In [ ]:
%%writefile /content/gigahands_training/hand_tracking/engine.py
"""Epoch runners shared by training, validation and the model-free anchor baseline."""
import time
import torch
from .constants import NUM_CAMERAS, CALIBRATION_PARAMS
from .contracts import model_inputs
from .data import trim_padding
from .objectives import PoseTotals, WeightedMean, pose_loss, calibration_loss


def to_device(batch, device):
    return {key: value.to(device, non_blocking=True) for key, value in batch.items()}


def batch_loss(model, batch, output, bone_weight, calibration_weight, world_weight=0.):
    """Pose loss over every query time (target frame, plus world_weight times the world-frame
    loss, which keeps a weak pull toward true world coordinates) plus the weighted
    calibration auxiliary loss.

    Returns (loss, calibration loss, number of known camera-event targets). The simulator's
    true camera correction supervises every accepted event's calibration (never an input);
    padding, dropped captures and real captures (NaN targets) carry none.
    """
    config = model.config
    loss = pose_loss(output.pose, batch['target'], batch['target_mask'], batch['world_unit_cm'], bone_weight)
    if world_weight:
        loss = loss+world_weight*pose_loss(output.pose, batch['target_world'], batch['target_mask'],
                                           batch['world_unit_cm'], bone_weight)
    if output.calibration is None:                                     # a model without camera correction
        return loss, loss.new_zeros(()), loss.new_zeros((), dtype=torch.long)
    target = torch.where(output.accepted[:,:,None,None], batch['calibration_target'][:,None], float('nan'))
    target = target.reshape(-1, NUM_CAMERAS, CALIBRATION_PARAMS)
    auxiliary = calibration_loss(output.calibration.reshape(-1, NUM_CAMERAS, CALIBRATION_PARAMS), target,
                                 config.calibration_rotation_deg, config.calibration_position)
    if calibration_weight and config.calibration_head:
        loss = loss+calibration_weight*auxiliary
    return loss, auxiliary, torch.isfinite(target).all(-1).sum()


# Mixed precision: consecutive overflowing steps GradScaler may skip before training stops.
MAX_SKIPPED_STEPS = 50


def optimizer_step(model, optimizer, scaler, loss):
    """Backward, clip to norm 1 and step; returns whether the gradients were finite.

    With mixed precision a scaled fp16 gradient overflows now and then: GradScaler then
    skips the step and lowers its scale, so a non-finite norm is not an error there.
    In full precision it is (a NaN from the model), so it raises."""
    optimizer.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1., error_if_nonfinite=not scaler.is_enabled())
    scaler.step(optimizer)
    scaler.update()
    return bool(torch.isfinite(norm))


def run_epoch(model, loader, device, optimizer=None, scaler=None, limit=0, bone_weight=.1, log_every=50,
              calibration_weight=0., world_weight=0.):
    """One pass over loader; trains when an optimizer is given. Metrics use the final query."""
    training = optimizer is not None
    model.train(training)
    pose, loss_mean, calibration = PoseTotals(), WeightedMean(), WeightedMean()
    started, batches, skipped, consecutive = time.monotonic(), 0, 0, 0
    for batch in loader:
        batch = to_device(trim_padding(batch), device)
        with torch.set_grad_enabled(training), \
                torch.autocast(device_type=device.type, enabled=training and device.type == 'cuda'):
            output = model(*model_inputs(batch), return_details=True)
            loss, auxiliary, known = batch_loss(model, batch, output, bone_weight, calibration_weight, world_weight)
        if not torch.isfinite(loss):
            raise FloatingPointError('Nonfinite loss')
        if training:
            if optimizer_step(model, optimizer, scaler, loss):
                consecutive = 0
            else:
                skipped, consecutive = skipped+1, consecutive+1
                if consecutive >= MAX_SKIPPED_STEPS:
                    raise FloatingPointError(f'Non-finite gradients in {consecutive} consecutive steps '
                                             f'(grad scale {scaler.get_scale():.3g}): not an fp16 overflow')
        pose.add(output.pose, batch)
        loss_mean.add(loss.item(), len(output.pose))
        # Match calibration_loss: one item per known camera and event.
        calibration.add(auxiliary.item(), known.item())
        batches += 1
        if batches % log_every == 0:
            print(f"{'train' if training else 'val'} batch={batches} samples={pose.samples} "
                  f"MPJPE={pose.mpjpe_mm:.2f}mm" + (f' skipped_steps={skipped}' if skipped else ''), flush=True)
        if limit and batches >= limit:
            break
    summary = pose.summary()
    return dict(loss=loss_mean.mean, mpjpe_mm=summary['mpjpe_mm'], pck20=summary['pck20'],
                mpjpe_world_mm=summary['mpjpe_world_mm'],
                calibration_mse=calibration.mean, samples=summary['samples'], joints=summary['joints'],
                skipped_steps=skipped,
                batches=batches, seconds=time.monotonic()-started)


def evaluate_anchor(loader, device, config):
    """Final-query metrics of the model-free anchor: HandLite with config's geometry, no
    calibration head and its zero-initialised residual head, so the pose is the anchor."""
    from dataclasses import replace
    from .lite import HandLite
    model = HandLite(replace(config, calibration_head=False)).to(device).eval()
    pose = PoseTotals()
    with torch.inference_mode():
        for batch in loader:
            batch = to_device(trim_padding(batch), device)
            pose.add(model(*model_inputs(batch)), batch)
    return pose.summary()


In [ ]:
%%writefile /content/gigahands_training/hand_tracking/events.py
"""Camera events: acceptance, latest-ray selection, triangulation samples and the
query-time anchor of HandLite. No learned parameters."""
import torch
import torch.nn.functional as F
from .constants import (NUM_CAMERAS, NUM_HANDS, NUM_JOINTS, HAND_JOINTS, MODEL_CHANNELS, RAY_ORIGIN,
                        RAY_DIRECTION, MISS_SCALE, TIME_UNIT_S, ARRIVAL_TOLERANCE_S, REPEAT_TOLERANCE_S,
                        MIN_TIME_STD_S, MAX_EXTRAPOLATION_S, MIN_RAY_DEPTH)
from .contracts import EventBatch, RawEvents
from .geometry import triangulate, robust_triangulate, apply_calibration



def accepted_captures(camera, capture, present):
    """Keep strictly newer captures per camera in arrival order; padding is ignored.

    This is the batch equivalent of EventStream's per-camera high-water mark: an event is
    dropped when an earlier present event of its camera has a capture at least as new.
    Pairwise [B,E,E] instead of a running maximum (cummax has no ONNX operator).
    Comparing the original dtype preserves float64 stream timestamp precision.
    """
    present = present.bool() & (camera >= 0) & (camera < NUM_CAMERAS) & torch.isfinite(capture)
    index = torch.arange(camera.shape[1], device=camera.device)
    earlier = index[None,:,None] > index[None,None,:]                                   # [1,E(i),E(j)]: j before i
    superseded = (earlier & present[:,None,:] & (camera[:,:,None] == camera[:,None,:])
                  & (capture[:,None,:] >= capture[:,:,None])).any(-1)
    return present & ~superseded


def make_events(features, valid, camera, capture, arrival, present) -> EventBatch:
    present = accepted_captures(camera, capture, present)
    return _pack_events(features, valid, camera, capture, arrival, present)


def _pack_events(features, valid, camera, capture, arrival, present) -> EventBatch:
    """Canonical tensor layout after the caller has resolved event acceptance."""
    valid = valid.bool() & present[...,None,None]
    raw = torch.where(valid[...,None], features[...,:MODEL_CHANNELS].float(), 0.)
    return dict(raw=raw, valid=valid, camera=camera.long(), capture=capture.float(), arrival=arrival.float(),
                present=present.bool())


def relative_events(raw: RawEvents, origin: float) -> EventBatch:
    """Convert one stream's absolute float64 timestamps into a relative model batch."""
    present = torch.ones_like(raw['camera'], dtype=torch.bool)[None]
    # Stream buffers contain only accepted frames; do not rescan history per query.
    return _pack_events(raw['features'][None], raw['valid'][None], raw['camera'][None],
                       (raw['capture']-origin)[None], (raw['arrival']-origin)[None], present)


def _gather(values, index):
    """values [B,E,...], index [B,...] -> values at index along the event axis."""
    batch = torch.arange(values.shape[0], device=values.device).reshape(-1, *[1]*(index.ndim-1))
    return values[batch, index]


def _up_to(events, targets):
    """[B,T,E]: events at or before each target in input (arrival) order. Input order,
    not arrival-time comparison, defines causality, so events with equal arrival times
    see each other exactly as when pushed one by one."""
    index = torch.arange(events['camera'].shape[1], device=events['camera'].device)
    return events['present'][:,None] & (index[None,None] <= targets[None,:,None])


def select_slots(events, query, per_camera, span_s):
    """Slots [B,Q,3K]: event indices in arrival order (padding first), and validity. Each
    camera's K latest accepted events that arrived by the query and were captured within
    span_s before it: a fixed-size input whatever the frame rate."""
    b, e = events['camera'].shape
    index = torch.arange(e, device=query.device)
    usable = (events['present'][:, None] & (events['arrival'][:, None] <= query[..., None]+ARRIVAL_TOLERANCE_S)
              & (events['capture'][:, None] >= query[..., None]-span_s))                    # [B,Q,E]
    chosen = []
    for camera in range(NUM_CAMERAS):
        score = torch.where(usable & (events['camera'][:, None] == camera), index, -1)
        if e < per_camera:
            score = F.pad(score, (0, per_camera-e), value=-1)
        chosen.append(score.topk(per_camera, dim=-1).values)
    slots = torch.cat(chosen, -1).sort(-1).values
    return slots.clamp_min(0), slots >= 0


def latest_rays(events, targets, max_age):
    """Each camera's most recently captured ray (among events up to the target, captured
    within max_age of its arrival) for every target event: origins/directions
    [B,T,3,2,21,3], mask [B,T,3,2,21], capture [B,T,3]. A late, older capture never
    replaces a newer one. Frames without any detected joint still count as the latest."""
    arrival = events['arrival']; capture = events['capture']
    target_arrival = arrival[:,targets]
    ok = _up_to(events, targets) & (capture[:,None] >= target_arrival[...,None]-max_age)     # [B,T,E]
    scores = torch.stack([torch.where(ok & (events['camera'][:,None] == c), capture[:,None], float('-inf'))
                          for c in range(NUM_CAMERAS)], -1)                              # [B,T,E,3]
    has = torch.isfinite(scores.amax(2)); pick = scores.argmax(2)                        # [B,T,3]
    raw = _gather(events['raw'], pick)                                                   # [B,T,3,2,21,C]
    mask = _gather(events['valid'], pick) & has[...,None,None]
    return raw[...,RAY_ORIGIN], raw[...,RAY_DIRECTION], mask, torch.where(has, _gather(capture, pick), 0.)


def sample_points(origins, directions, mask, capture, params=None, outlier_ratio=None):
    """Triangulate each target's latest-ray sample, optionally calibrated with that
    target's params [B,T,3,6]. Returns point [B,T,2,21,3], ok [B,T,2,21], stamp (mean
    capture of the used rays) [B,T,2,21] and the calibrated rays. outlier_ratio drops one
    inconsistent ray (geometry.robust_triangulate)."""
    b, t = origins.shape[:2]
    if params is not None:
        origins, directions = apply_calibration(params.reshape(b*t,3,6), origins.reshape(b*t,1,*origins.shape[2:]),
                                                directions.reshape(b*t,1,*directions.shape[2:]))
        origins, directions = origins.reshape(b,t,*origins.shape[2:]), directions.reshape(b,t,*directions.shape[2:])
    origins = torch.where(mask[...,None], origins, 0.); directions = torch.where(mask[...,None], directions, 0.)
    order = (0,1,3,4,2,5)
    used = mask.permute(0,1,3,4,2)
    if outlier_ratio:
        point, ok, used = robust_triangulate(origins.permute(order), directions.permute(order), used, outlier_ratio)
    else:
        point, ok = triangulate(origins.permute(order), directions.permute(order), used)
    weight = used.float()
    stamp = (capture[:,:,None,None,:]*weight).sum(-1)/weight.sum(-1).clamp_min(1)
    return point, ok, stamp, (origins, directions)


def _miss(point, ok, origin, direction, use):
    """Perpendicular vectors (x MISS_SCALE) from each ray to the point, zero where unusable."""
    relative = point-origin
    miss = relative-direction*(relative*direction).sum(-1, keepdim=True)
    return torch.where((use & ok)[...,None], miss*MISS_SCALE, 0.)


def _gather_joint(values, pick):
    """values [B,E,2,21,C], pick [B,Q,2,21] event index per joint -> [B,Q,2,21,C]."""
    b, e = values.shape[:2]; q = pick.shape[1]; c = values.shape[-1]
    flat = values.reshape(b, e, HAND_JOINTS, c).permute(0,2,1,3)                        # [B,42,E,C]
    index = pick.reshape(b, q, HAND_JOINTS).permute(0,2,1)[...,None].expand(-1,-1,-1,c) # [B,42,Q,C]
    return flat.gather(2, index).permute(0,2,1,3).reshape(b, q, NUM_HANDS, NUM_JOINTS, c)


def query_anchor(point, ok, stamp, arrival, present, query, lookback_s, hold_s, single=None):
    """Anchor at each query time from the samples of events that arrived by then.

    point/ok/stamp [B,E,2,21,...] per event sample, arrival/present [B,E], query [B,Q].
    A line fitted to the distinct samples captured within lookback_s is extrapolated to
    the query (at most MAX_EXTRAPOLATION_S); otherwise the latest sample within hold_s is
    held.
    single = (origin, direction, has, capture, reference, known) (see single_ray_anchor): a
    joint the latest event did not triangulate but that has a ray is anchored on the ray
    closest to its anchor, else its reference, else its hand's mean reference, at that depth.
    Joints without a sample use the mean anchor of the same hand, else the origin.
    Returns anchor [B,Q,2,21,3] and flags [B,Q,2,21,5]: latest event's sample valid, joint
    anchor exists, hand anchor exists, age of the latest sample (TIME_UNIT_S), extrapolated;
    with single, a sixth flag marks single-ray anchors (age is then the ray's).
    """
    e = arrival.shape[1]; index = torch.arange(e, device=arrival.device)
    available = present[:,None] & (arrival[:,None] <= query[...,None]+ARRIVAL_TOLERANCE_S)  # [B,Q,E]
    latest_event = torch.where(available, index, -1).amax(-1)                          # [B,Q]
    now = _gather(ok, latest_event.clamp_min(0)) & (latest_event >= 0)[...,None,None]
    tau = stamp[:,None]-query[:,:,None,None,None]                                      # [B,Q,E,2,21]
    usable = available[...,None,None] & ok[:,None] & (-tau <= hold_s)
    latest = torch.where(usable, index[None,None,:,None,None], -1).amax(2)             # [B,Q,2,21]
    joint = latest >= 0; pick = latest.clamp_min(0)
    anchor = _gather_joint(point, pick)
    age = torch.where(joint, query[...,None,None]-_gather_joint(stamp[...,None], pick)[...,0], 0.)
    moving = torch.zeros_like(joint)
    if lookback_s > 0:
        use = usable & (-tau <= lookback_s)
        # A sample repeated over events (a joint the newest event did not see) counts once.
        same = ((stamp[:,:,None]-stamp[:,None]).abs() <= REPEAT_TOLERANCE_S) & ok[:,:,None] & ok[:,None]
        count = torch.einsum('bsuhj,btuhj->btshj', same.float(), use.float())
        w = torch.where(use, 1/count.clamp_min(1), 0.)
        total = w.sum(2)
        mean_tau = (w*tau).sum(2)/total.clamp_min(1e-12)
        mean_point = (w[...,None]*point[:,None]).sum(2)/total.clamp_min(1e-12)[...,None]
        centred = tau-mean_tau[:,:,None]
        var_tau = (w*centred.square()).sum(2)
        slope = (w[...,None]*centred[...,None]*(point[:,None]-mean_point[:,:,None])).sum(2)
        slope = slope/var_tau.clamp_min(1e-12)[...,None]
        moving = (total >= 2-1e-6) & (var_tau > MIN_TIME_STD_S**2*total)
        anchor = torch.where(moving[...,None], mean_point+slope*(-mean_tau).clamp(0., MAX_EXTRAPOLATION_S)[...,None], anchor)
    extra = []
    if single is not None:
        anchor, joint, moving, age, ray = single_ray_anchor(anchor, joint, moving, age, now, query, *single)
        extra = [ray.float()]
    weight = joint[...,None].float()
    hand_mean = (anchor*weight).sum(-2, keepdim=True)/weight.sum(-2, keepdim=True).clamp_min(1)
    hand = joint.any(-1, keepdim=True).expand_as(joint)
    anchor = torch.where(joint[...,None], anchor, hand_mean)
    return anchor, torch.stack((now.float(), joint.float(), hand.float(), age/TIME_UNIT_S, moving.float(), *extra), -1)


def single_ray_anchor(anchor, joint, moving, age, now, query, origin, direction, has, capture, reference, known):
    """Anchor joints the latest event did not triangulate (now False) on one of their rays.
    origin/direction [B,Q,2,21,C,3], has/capture [B,Q,2,21,C]: each camera's latest ray;
    reference [B,Q,2,21,3]/known [B,Q,2,21]: a remembered triangulation. The reference point
    is the joint's anchor if it has one, else its remembered triangulation, else the mean of
    its hand's; the ray with the smallest angular residual to it is used (with two rays that
    do not agree, the one not contradicting the past), at the reference's depth. Without a
    reference the joint keeps its anchor."""
    reference = torch.where(joint[...,None], anchor, reference)
    known = joint | known
    weight = known[...,None].float()
    hand = (reference*weight).sum(-2, keepdim=True)/weight.sum(-2, keepdim=True).clamp_min(1)
    reference = torch.where(known[...,None], reference, hand)
    known = known.any(-1, keepdim=True).expand_as(known)
    direction = F.normalize(direction, dim=-1)
    relative = reference[...,None,:]-origin
    along = (relative*direction).sum(-1)                                                # [B,Q,2,21,C]
    residual = (relative-along[...,None]*direction).norm(dim=-1)/along.clamp_min(1e-6)
    best = torch.where(has, residual, float('inf')).argmin(-1, keepdim=True)
    pick = lambda value: value.gather(-1, best).squeeze(-1)
    origin, direction = (value.gather(-2, best[...,None].expand(*best.shape, 3)).squeeze(-2)
                         for value in (origin, direction))
    ray = has.any(-1) & ~now & known
    depth = pick(along)[...,None].clamp_min(MIN_RAY_DEPTH)
    anchor = torch.where(ray[...,None], origin+direction*depth, anchor)
    age = torch.where(ray, query[...,None,None]-pick(capture), age)
    return anchor, joint | ray, moving & ~ray, age, ray


In [ ]:
%%writefile /content/gigahands_training/hand_tracking/geometry.py
"""Differentiable multi-view ray geometry: triangulation (optionally dropping one outlier
ray) and camera corrections. No learned parameters."""
import torch
from .constants import MIN_RAY_SIN2, RAY_RESIDUAL_FLOOR, OUTLIER_MARGIN


def cross3(a, b):
    """Cross product of [...,3] vectors (torch.linalg.cross has no ONNX operator)."""
    a0, a1, a2 = a.unbind(-1); b0, b1, b2 = b.unbind(-1)
    return torch.stack((a1*b2-a2*b1, a2*b0-a0*b2, a0*b1-a1*b0), -1)


def solve3(a, b):
    """a [...,3,3] invertible, b [...,3] -> a^-1 b by the adjugate (Cramer's rule).

    Exact algebra for 3x3 and exportable (torch.linalg.solve has no ONNX operator);
    callers substitute the identity for singular systems, as for linalg.solve.
    """
    r0, r1, r2 = a.unbind(-2)
    c0, c1, c2 = cross3(r1, r2), cross3(r2, r0), cross3(r0, r1)
    det = (r0*c0).sum(-1, keepdim=True)
    return (b[...,0:1]*c0+b[...,1:2]*c1+b[...,2:3]*c2)/det


def _rays(origins, directions, mask):
    """Usable rays: mask, unit directions and origins (zero where unusable), projections
    I - d d^T [...,C,3,3] (zero where unusable) and pairwise sin^2 of ray angles [...,C,C]."""
    mask = (mask.bool() & torch.isfinite(origins).all(-1) & torch.isfinite(directions).all(-1)
            & (directions.norm(dim=-1) > 1e-8))
    d = torch.where(mask[...,None], directions, 0.); o = torch.where(mask[...,None], origins, 0.)
    d = d/d.norm(dim=-1, keepdim=True).clamp_min(1e-8)
    eye = torch.eye(3, dtype=d.dtype, device=d.device)
    projection = (eye - d[...,:,None]*d[...,None,:])*mask[...,None,None]
    # Widest pairwise ray angle instead of eigvalsh: batched CUDA eigh can fail to
    # converge on the empty/single-ray matrices (repeated eigenvalues) present here.
    pair = cross3(d[...,:,None,:], d[...,None,:,:]).square().sum(-1)
    return mask, d, o, projection, pair


def _point(mask, d, o, projection, pair):
    """Least-squares point of the rays in mask (a subset of _rays' rays) and its validity."""
    used = projection*mask[...,None,None]
    return _solve(used.sum(-3), (used @ o[...,None]).sum(-3).squeeze(-1), mask, d, o, pair)


def _solve(a, rhs, mask, d, o, pair):
    """Point from the normal equations a x = rhs of the rays in mask; d, o, pair may
    broadcast over leading axes of mask (leave-one-out problems share them)."""
    eye = torch.eye(3, dtype=d.dtype, device=d.device)
    sin2 = (pair*(mask[...,:,None] & mask[...,None,:])).amax((-2,-1))
    valid = (mask.sum(-1) >= 2) & (sin2 > MIN_RAY_SIN2)
    point = solve3(torch.where(valid[...,None,None], a, eye), rhs)
    # A point behind any contributing camera means inconsistent rays (outliers), not a hand.
    depth = ((point[...,None,:]-o)*d).sum(-1)
    valid = valid & ((depth > 0) | ~mask).all(-1)
    return torch.where(valid[...,None], point, 0.), valid


def triangulate(origins, directions, mask):
    """Least-squares point closest to the masked rays.

    origins/directions [...,C,3], mask [...,C] -> point [...,3], valid [...].
    Needs two or more non-parallel rays with positive depths; otherwise point is 0 and
    valid is False.
    """
    return _point(*_rays(origins, directions, mask))


def ray_residuals(point, origins, directions):
    """Angular residual (miss distance / depth) of each ray [...,C] to point [...,3]."""
    d = directions/directions.norm(dim=-1, keepdim=True).clamp_min(1e-8)
    relative = point[...,None,:]-origins
    depth = (relative*d).sum(-1)
    return (relative-depth[...,None]*d).norm(dim=-1)/depth.clamp_min(1e-6)


def robust_triangulate(origins, directions, mask, outlier_ratio):
    """triangulate, rejecting at most one outlier ray among three or more.

    For each ray k: e = its residual to the point of the other rays, s = their largest
    residual to that point. The ray with the largest e/max(s, RAY_RESIDUAL_FLOOR) is
    dropped when that ratio exceeds outlier_ratio and OUTLIER_MARGIN times the next ray's:
    the other rays agree and it does not. When the next ray's ratio is within that margin
    the sample is ambiguous and invalid: a ray displaced within the epipolar plane of
    another camera agrees with that camera, so pairs (0,1) and (0,2) can both agree while
    (1,2) do not, and geometry cannot tell whether 1 or 2 is wrong. A camera whose
    calibration is off looks like an outlier, so apply this to corrected rays.
    The leave-one-out points share the rays' preparation and are solved in one batch.
    Returns point, valid and the rays used [...,C].
    """
    mask, d, o, projection, pair = _rays(origins, directions, mask)
    # Normal equations of all rays, and of all but ray k by removing that ray's term
    # (projection is zero for unusable rays, so their "removal" changes nothing).
    term = (projection @ o[...,None]).squeeze(-1)                                        # [...,C,3]
    a, rhs = projection.sum(-3), term.sum(-2)
    point, valid = _solve(a, rhs, mask, d, o, pair)
    count = mask.shape[-1]
    others = ~torch.eye(count, dtype=torch.bool, device=mask.device)                    # [K,C]: all but k
    keep = mask[...,None,:] & others                                                    # [...,K,C]
    loo_point, loo_valid = _solve(a[...,None,:,:]-projection, rhs[...,None,:]-term, keep, d[...,None,:,:],
                                  o[...,None,:,:], pair[...,None,:,:])
    relative = loo_point[...,None,:]-o[...,None,:,:]                                    # [...,K,C,3]
    along = (relative*d[...,None,:,:]).sum(-1)
    residual = (relative-along[...,None]*d[...,None,:,:]).norm(dim=-1)/along.clamp_min(1e-6)   # [...,K,C]
    spread = torch.where(keep, residual, 0.).amax(-1)
    own = residual.diagonal(dim1=-2, dim2=-1)                                           # [...,K]
    ratio = torch.where(loo_valid & mask & (mask.sum(-1, keepdim=True) >= 3), own/spread.clamp_min(RAY_RESIDUAL_FLOOR), 0.)
    top = ratio.topk(min(2, count), dim=-1)
    best, dropped = top.values[...,0], top.indices[...,0]
    second = top.values[...,1] if count > 1 else torch.zeros_like(best)
    pick = lambda value: value.gather(-1, dropped[...,None]).squeeze(-1)
    best_point = loo_point.gather(-2, dropped[...,None,None].expand(*dropped.shape, 1, 3)).squeeze(-2)
    suspect = best > outlier_ratio
    ambiguous = suspect & (best < OUTLIER_MARGIN*second)
    reject = suspect & ~ambiguous
    valid = torch.where(reject, pick(loo_valid), valid) & ~ambiguous
    point = torch.where(valid[...,None], torch.where(reject[...,None], best_point, point), 0.)
    return point, valid, mask & ~(reject[...,None] & (torch.arange(count, device=mask.device) == dropped[...,None]))


def _sinc(theta, theta2, small):
    """sin(theta)/theta, Taylor below the `small` threshold (finite value and gradient at 0)."""
    safe = torch.where(small, torch.ones_like(theta), theta)
    return torch.where(small, 1-theta2/6, torch.sin(safe)/safe)


def rodrigues(w):
    """Rotation vectors [...,3] -> matrices [...,3,3]; smooth and differentiable at zero.

    R = I + sinc(t) [w]x + (1-cos t)/t^2 [w]x^2 with (1-cos t)/t^2 = sinc(t/2)^2/2, which
    has no float32 cancellation. No epsilon inside a sqrt: ONNX export folded it away.
    """
    theta2 = w.square().sum(-1)[...,None,None]
    small = theta2 < 1e-8
    theta = torch.where(small, torch.ones_like(theta2), theta2).sqrt()
    a = _sinc(theta, theta2, small)
    b = .5*_sinc(theta/2, theta2/4, small).square()
    zero = torch.zeros_like(w[...,0])
    skew = torch.stack((zero,-w[...,2],w[...,1], w[...,2],zero,-w[...,0], -w[...,1],w[...,0],zero), -1)
    skew = skew.reshape(*w.shape[:-1],3,3)
    eye = torch.eye(3, dtype=w.dtype, device=w.device)
    return eye+a*skew+b*(skew@skew)


def apply_calibration(params, origins, directions):
    """params [B,3,6] (rotation vector, translation per camera); rays [B,T,3,2,21,3].

    Rotates each camera's rays about its centre and shifts the centre, both in world axes.
    """
    rotation = rodrigues(params[...,:3])
    origins = origins+params[:,None,:,None,None,3:]
    directions = torch.einsum('bcxy,btchjy->btchjx', rotation, directions)
    return origins, directions


In [ ]:
%%writefile /content/gigahands_training/hand_tracking/layers.py
"""Network building blocks shared by the models: attention with an additive key bias,
pre-norm blocks and the per-event finger-token encoder. Linear/LayerNorm/GELU/MatMul/
Softmax only and tensors of at most 4 dimensions, so the graphs export to ONNX and ncnn."""
import torch
from torch import nn
import torch.nn.functional as F
from .constants import NUM_CAMERAS, NUM_HANDS, NUM_JOINTS, FINGERS, FINGER_JOINTS, MASK_OFF


def key_bias(valid):
    """Boolean key validity -> additive attention bias (0 or MASK_OFF)."""
    return (valid.float()-1.)*-MASK_OFF


class Attention(nn.Module):
    """Multi-head attention with an additive bias; 4-D tensors at most (ncnn limit)."""

    def __init__(self, dim, heads, dropout):
        super().__init__()
        self.heads, self.dropout = heads, dropout
        self.query, self.key, self.value, self.out = (nn.Linear(dim, dim) for _ in range(4))

    def forward(self, x, keys, bias=None):
        """x [N,L,d], keys [N,S,d], bias broadcastable to [N,heads,L,S] -> [N,L,d]."""
        n, l, d = x.shape
        s, h = keys.shape[1], self.heads
        q = self.query(x).reshape(n, l, h, d//h).permute(0, 2, 1, 3)
        k = self.key(keys).reshape(n, s, h, d//h).permute(0, 2, 3, 1)
        v = self.value(keys).reshape(n, s, h, d//h).permute(0, 2, 1, 3)
        scores = (q @ k)*(d//h)**-.5
        if bias is not None:
            scores = scores+bias
        weights = F.dropout(scores.softmax(-1), self.dropout, self.training)
        return self.out((weights @ v).permute(0, 2, 1, 3).reshape(n, l, d))


class Block(nn.Module):
    """Pre-norm: optional cross-attention to keys, self-attention, feed-forward. dropout on
    the residual branches, attention_dropout on the attention weights."""

    def __init__(self, dim, heads, dropout, cross=True, attention_dropout=0.):
        super().__init__()
        self.cross = Attention(dim, heads, attention_dropout) if cross else None
        if cross:
            self.norm_query, self.norm_keys = nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.norm_self, self.norm_ff = nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.attention = Attention(dim, heads, attention_dropout)
        self.feedforward = nn.Sequential(nn.Linear(dim, 2*dim), nn.GELU(), nn.Linear(2*dim, dim))
        self.drop = nn.Dropout(dropout)

    def forward(self, x, keys=None, bias=None, self_bias=None):
        """bias masks the cross-attention keys, self_bias the self-attention ones."""
        if self.cross is not None:
            x = x+self.drop(self.cross(self.norm_query(x), self.norm_keys(keys), bias))
        y = self.norm_self(x)
        x = x+self.drop(self.attention(y, y, self_bias))
        return x+self.drop(self.feedforward(self.norm_ff(x)))


class EventEncoder(nn.Module):
    """features [N,2,21*F], camera one-hot [N,3] -> tokens [N,2*5,dim] (hand-major, one per
    finger) with finger_tokens, else [N,2,dim] (one per hand).

    Finger tokens share one MLP over the 5 joints (wrist and finger) of a group; a learned
    finger embedding inside the MLP lets each finger be read differently. Groups are taken
    with a constant 0/1 selection matrix (MatMul), which ncnn runs without Gather."""

    def __init__(self, dim, finger_tokens=True, joint_features=14):
        super().__init__()
        self.fingers, self.joint_features = finger_tokens, joint_features
        if finger_tokens:
            group = len(FINGER_JOINTS[0])
            select = torch.zeros(FINGERS*group, NUM_JOINTS)
            select[torch.arange(FINGERS*group), torch.tensor(FINGER_JOINTS).flatten()] = 1.
            self.register_buffer('select', select, persistent=False)
            self.joints = nn.Sequential(nn.Linear(group*joint_features, dim), nn.GELU(), nn.Linear(dim, dim))
            self.finger = nn.Parameter(torch.randn(FINGERS, dim)*.02)
        else:
            self.joints = nn.Sequential(nn.Linear(NUM_JOINTS*joint_features, dim), nn.GELU(), nn.Linear(dim, dim))
        self.hand = nn.Parameter(torch.randn(NUM_HANDS, dim)*.02)
        self.camera = nn.Linear(NUM_CAMERAS, dim, bias=False)
        self.norm = nn.LayerNorm(dim)

    def forward(self, features, camera):
        if not self.fingers:
            return self.norm(self.joints(features)+self.hand+self.camera(camera)[:, None])
        n = features.shape[0]
        groups = self.select@features.reshape(n, NUM_HANDS, NUM_JOINTS, self.joint_features)   # [N,2,5*5,F]
        groups = groups.reshape(n, NUM_HANDS, FINGERS, -1)                                     # [N,2,5,5*F]
        x = self.joints[2](self.joints[1](self.joints[0](groups)+self.finger))
        x = x+self.hand[:, None]+self.camera(camera)[:, None, None]
        return self.norm(x).reshape(n, NUM_HANDS*FINGERS, -1)


In [ ]:
%%writefile /content/gigahands_training/hand_tracking/lite.py
"""HandLite: a fixed-size, export-friendly pose model for asynchronous camera events.

Split so that deployment runs three small fixed-shape networks (ONNX/ncnn) around plain
geometry (lite_runtime, numpy):
  per event, once, on arrival:
    geometry  latest ray of each camera; nominal triangulation; the event's own ray miss
    encoder   21 joint features per hand -> per hand one token per finger (wrist and its
              4 joints), or one token without finger_tokens            [event_tokens, dim]
  per query at time t:
    slots     each camera's slots_per_camera latest events captured within event_span_s
    calibrator  tokens pooled per camera -> camera correction                 [3, 6]
    geometry  corrected triangulation of every slot's rays (one outlier ray dropped); line fit
              to t -> anchor + flags;
              a joint seen by one camera: on its newest ray at a remembered depth
    decoder   42 joint queries (anchor) attend to the event_tokens*3*K tokens, each tagged with its
              capture-to-query gap -> offset [42, 3]
The pose is anchor + offset. Networks use only Linear/LayerNorm/GELU/MatMul/Softmax and
additive float masks. forward() computes exactly what LiteStream computes event by event.
"""
from dataclasses import dataclass, replace
import torch
from torch import nn
import torch.nn.functional as F
from .config import LiteConfig
from .constants import (NUM_CAMERAS, NUM_HANDS, NUM_JOINTS, HAND_JOINTS, FINGERS, FINGER_JOINTS, FEATURE_CHANNELS,
                        CALIBRATION_PARAMS, UV, RAY_ORIGIN, RAY_DIRECTION, DELAY, TIME_UNIT_S, ARRIVAL_TOLERANCE_S, MASK_OFF)
from .contracts import ModelOutput
from .events import (_gather, _gather_joint, _miss, latest_rays, make_events, query_anchor, relative_events,
                     select_slots,
                     sample_points)
from .layers import Attention, Block, EventEncoder, key_bias
from .stream import EventStream

# Per joint: u,v (2), ray origin (3) and direction (3), delay (1), miss vector (3), seen (1),
# triangulated with another camera (1).
JOINT_FEATURES = 14
HAND_FEATURES = NUM_JOINTS*JOINT_FEATURES
# Margin on anchor_hold_s when dropping slots too old for the anchor (float32 relative times).
REACH_MARGIN_S = 1e-3


class Calibrator(nn.Module):
    """tokens [N,T,dim], pool bias [N,3,T] (camera c's valid tokens 0, others MASK_OFF)
    -> correction [N,3,6] (rotation vector rad, shift world units); zero until trained."""

    def __init__(self, dim, heads, dropout, scale, attention_dropout=0.):
        super().__init__()
        self.query = nn.Parameter(torch.randn(NUM_CAMERAS, dim)*.02)
        self.null = nn.Parameter(torch.zeros(1, dim))
        self.norm = nn.LayerNorm(dim)
        self.pool = Attention(dim, heads, attention_dropout)
        self.mix = Block(dim, heads, dropout, cross=False, attention_dropout=attention_dropout)
        self.output = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, dim), nn.GELU(), nn.Linear(dim, CALIBRATION_PARAMS))
        nn.init.zeros_(self.output[-1].weight)
        nn.init.zeros_(self.output[-1].bias)
        self.register_buffer('scale', scale)

    def forward(self, tokens, pool_bias):
        n, _, d = tokens.shape
        keys = torch.cat((self.null.expand(n, 1, d), self.norm(tokens)), 1)
        bias = torch.cat((torch.zeros_like(pool_bias[:, :, :1]), pool_bias), -1)[:, None]
        pooled = self.pool(self.query.expand(n, NUM_CAMERAS, d), keys, bias)
        return self.output(self.mix(pooled))*self.scale


class Decoder(nn.Module):
    """tokens [N,T,dim], token validity [N,T] (1/0), capture-to-query gaps [N,T] (TIME_UNIT_S),
    anchor features [N,42,config.anchor_features] -> offset [N,42,3] from the anchor; zero until trained.
    An invalid token's score is exactly MASK_OFF: the learned time bias cannot revive it.
    The gap sets how much a token is attended (time_bias) and, with gap_embedding, is added
    to the token itself so values carry their age; zero-initialised, so it starts inert."""

    def __init__(self, dim, heads, blocks, dropout, gap_embedding=True, anchor_features=9, attention_dropout=0.):
        super().__init__()
        self.joints = nn.Parameter(torch.randn(HAND_JOINTS, dim)*.02)
        self.anchor = nn.Sequential(nn.Linear(anchor_features, dim), nn.GELU(), nn.Linear(dim, dim))
        self.null = nn.Parameter(torch.zeros(1, dim))
        self.time_bias = nn.Sequential(nn.Linear(1, 16), nn.GELU(), nn.Linear(16, heads))
        self.gap = nn.Sequential(nn.Linear(1, 16), nn.GELU(), nn.Linear(16, dim)) if gap_embedding else None
        if self.gap is not None:
            nn.init.zeros_(self.gap[-1].weight)
            nn.init.zeros_(self.gap[-1].bias)
        self.blocks = nn.ModuleList([Block(dim, heads, dropout, attention_dropout=attention_dropout)
                                     for _ in range(blocks)])
        self.output = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, dim), nn.GELU(), nn.Linear(dim, 3))
        nn.init.zeros_(self.output[-1].weight)
        nn.init.zeros_(self.output[-1].bias)

    def forward(self, tokens, token_valid, gaps, anchor_features):
        n, _, d = tokens.shape
        x = self.joints+self.anchor(anchor_features)
        if self.gap is not None:
            tokens = tokens+self.gap(gaps[..., None])*token_valid[..., None]                   # padding stays zero
        keys = torch.cat((self.null.expand(n, 1, d), tokens), 1)
        valid = token_valid[:, None]
        bias = self.time_bias(gaps[..., None]).permute(0, 2, 1)*valid+(valid-1.)*-MASK_OFF      # [N,h,T]
        bias = torch.cat((torch.zeros_like(bias[..., :1]), bias), -1)[:, :, None]             # [N,h,1,1+T]
        for block in self.blocks:
            x = block(x, keys, bias)
        return self.output(x)


@dataclass
class LiteEncoded:
    """Per-event results that never change after arrival. Times on the batch origin
    (stream storage: absolute float64)."""
    origin: torch.Tensor     # [B,E,3,2,21,3] each camera's latest ray at the event's arrival
    direction: torch.Tensor  # [B,E,3,2,21,3]
    mask: torch.Tensor       # [B,E,3,2,21]
    capture: torch.Tensor    # [B,E,3] capture time of those rays
    evident: torch.Tensor    # [B,E] some own joint triangulated with another camera
    tokens: torch.Tensor     # [B,E,event_tokens,dim]
    point: torch.Tensor      # [B,E,2,21,3] nominal triangulation of those rays
    ok: torch.Tensor         # [B,E,2,21]
    stamp: torch.Tensor      # [B,E,2,21] mean capture time of the rays used

    def map(self, fn):
        return LiteEncoded(*(fn(value) for value in self.__dict__.values()))

    def append(self, other, dim=1):
        return LiteEncoded(*(torch.cat((a, b), dim) for a, b in zip(self.__dict__.values(), other.__dict__.values())))


def joint_features(raw, valid, point, ok):
    """[...,2,21,11] event features, own valid, nominal sample point/ok -> [...,2,21,14]."""
    origin, direction = raw[..., RAY_ORIGIN], raw[..., RAY_DIRECTION]
    miss = _miss(point, ok, origin, direction, valid)
    seen = (ok & valid).float()[..., None]
    x = torch.cat((raw[..., UV]*2-1, origin, direction, raw[..., DELAY]/TIME_UNIT_S, miss, valid.float()[..., None],
                   seen), -1)
    return torch.where(valid[..., None], x, 0.)


class HandLite(nn.Module):
    def __init__(self, config=None):
        super().__init__()
        self.config = config or LiteConfig()
        c = self.config
        scale = torch.tensor([c.calibration_rotation_deg*torch.pi/180]*3+[c.calibration_position]*3)
        self.encoder = EventEncoder(c.dim, c.finger_tokens, JOINT_FEATURES)
        self.calibrator = Calibrator(c.dim, c.heads, c.dropout, scale, c.attention_dropout) if c.calibration_head else None
        self.decoder = Decoder(c.dim, c.heads, c.blocks, c.dropout, c.gap_embedding, c.anchor_features, c.attention_dropout)

    @property
    def slots(self):
        return NUM_CAMERAS*self.config.slots_per_camera

    def encode_events(self, events, targets=None):
        """LiteEncoded of the target events (default: all), given every earlier event."""
        b, e = events['camera'].shape
        if targets is None:
            targets = torch.arange(e, device=events['camera'].device)
        t = targets.numel()
        with torch.autocast(device_type=events['raw'].device.type, enabled=False):
            origin, direction, mask, capture = latest_rays(events, targets, self.config.sample_max_age_s)
            point, ok, stamp, _ = sample_points(origin, direction, mask, capture)
            raw, valid = events['raw'][:, targets], events['valid'][:, targets]
            features = joint_features(raw, valid, point, ok)
        camera = F.one_hot(events['camera'][:, targets], NUM_CAMERAS).float()
        tokens = self.encoder(features.reshape(b*t, NUM_HANDS, HAND_FEATURES), camera.reshape(b*t, NUM_CAMERAS))
        evident = (ok & valid).flatten(2).any(-1) & events['present'][:, targets]
        return LiteEncoded(origin, direction, mask, capture, evident, tokens.reshape(b, t, *tokens.shape[1:]),
                           point, ok & events['present'][:, targets, None, None], stamp)

    def select(self, events, query):
        """Slots [B,Q,3K] and validity (events.select_slots)."""
        return select_slots(events, query, self.config.slots_per_camera, self.config.event_span_s)

    def decode_queries(self, events, encoded, query):
        """Pose [B,Q,2,21,3], calibration [B,Q,3,6] and whether a query had any slot [B,Q]."""
        c = self.config
        b, q = query.shape
        n, s, per = b*q, self.slots, c.event_tokens
        slot, valid = self.select(events, query)                                           # [B,Q,S]
        take = lambda value: _gather(value, slot).reshape(n, s, *value.shape[2:])
        camera, capture, arrival = take(events['camera']), take(events['capture']), take(events['arrival'])
        valid = valid.reshape(n, s)
        token_valid = valid.repeat_interleave(per, -1)
        # Padding slots point at event 0: zero what they carry, as the deployment runtime does.
        tokens = torch.where(token_valid[..., None], take(encoded.tokens).reshape(n, per*s, -1), 0.)
        token_camera = camera.repeat_interleave(per, -1)
        gaps = torch.where(valid, ((query.reshape(n, 1)-capture)/TIME_UNIT_S).clamp_min(0.), 0.)
        gaps = gaps.repeat_interleave(per, -1)
        cameras = torch.arange(NUM_CAMERAS, device=query.device)[:, None]
        if self.calibrator is not None:
            pool = key_bias((token_camera[:, None] == cameras) & token_valid[:, None])        # [N,3,2S]
            params = self.calibrator(tokens, pool).float()
            # A camera without triangulated evidence in its slots keeps the nominal calibration.
            informative = ((camera[:, None] == cameras) & (valid & take(encoded.evident))[:, None]).any(-1)
            params = torch.where(informative[..., None], params, 0.)
        else:
            params = tokens.new_zeros(n, NUM_CAMERAS, CALIBRATION_PARAMS)
        # The anchor's geometry needs only the slots that can reach it: a sample is at most as
        # new as its event's arrival, so a slot that arrived over anchor_hold_s before the query
        # is never used. Slots are in arrival order (padding first), so these are the last ones.
        reach = self.reaching_slots(valid, arrival, query.reshape(n, 1))
        near = lambda value: value[:, s-reach:]
        near_valid = near(valid)
        with torch.autocast(device_type=query.device.type, enabled=False):
            point, ok, stamp, rays = sample_points(near(take(encoded.origin)), near(take(encoded.direction)),
                                                   near(take(encoded.mask)), near(take(encoded.capture)),
                                                   params[:, None].expand(n, reach, -1, -1),
                                                   outlier_ratio=c.ray_outlier_ratio)
            ok = ok & near_valid[..., None, None]
            single = (self.single_rays(events, encoded, query, rays, near(take(encoded.mask)), near(take(encoded.capture)),
                                       near_valid) if c.anchor_ray_depth else None)
            anchor, flags = query_anchor(point, ok, stamp, near(arrival), near_valid, query.reshape(n, 1),
                                         c.anchor_lookback_s, c.anchor_hold_s, single=single)
        features = torch.cat((anchor, flags), -1).reshape(n, HAND_JOINTS, c.anchor_features)
        offset = self.decoder(tokens, token_valid.to(tokens.dtype), gaps, features)
        pose = anchor.reshape(n, HAND_JOINTS, 3)+offset.float()
        return (pose.reshape(b, q, NUM_HANDS, NUM_JOINTS, 3), params.reshape(b, q, NUM_CAMERAS, CALIBRATION_PARAMS),
                valid.any(-1).reshape(b, q))

    def reaching_slots(self, valid, arrival, query):
        """How many trailing slots any query of the batch needs for its anchor (at least 1:
        the latest slot gives the anchor's 'now' flag and single rays)."""
        reach = valid & (arrival >= query-self.config.anchor_hold_s-REACH_MARGIN_S)
        return max(int(reach.sum(-1).max()), 1)

    def single_rays(self, events, encoded, query, rays, mask, capture, valid):
        """query_anchor's single-ray inputs [N,1,2,21,...]: each camera's (calibrated) ray of
        each joint in the latest slot event's ray table [N,1,2,21,3(,3)], and the joint's
        newest nominal triangulation among all events arrived by the query and captured
        within anchor_depth_memory_s."""
        n = valid.shape[0]
        joints_last = lambda value: value.movedim(1, 3)                                       # camera axis after joint
        origins, directions = joints_last(rays[0][:, -1]), joints_last(rays[1][:, -1])        # [N,2,21,3,3]
        mask = joints_last(mask[:, -1] & valid[:, -1, None, None, None])                     # [N,2,21,3]
        ray_capture = capture[:, -1, None, None, :].expand_as(mask)
        index = torch.arange(events['camera'].shape[1], device=query.device)
        arrived = events['present'][:, None] & (events['arrival'][:, None] <= query[..., None]+ARRIVAL_TOLERANCE_S)
        remembered = (arrived[..., None, None] & encoded.ok[:, None]
                      & (encoded.stamp[:, None] >= query[..., None, None, None]-self.config.anchor_depth_memory_s))
        latest = torch.where(remembered, index[:, None, None], -1).amax(2)                  # [B,Q,2,21]
        reference = _gather_joint(encoded.point, latest.clamp_min(0))
        per_query = (origins, directions, mask, ray_capture,
                     reference.reshape(n, NUM_HANDS, NUM_JOINTS, 3), (latest >= 0).reshape(n, NUM_HANDS, NUM_JOINTS))
        return tuple(value[:, None] for value in per_query)                                 # query_anchor's Q=1

    def forward(self, event_features, event_valid, event_camera, event_capture, event_arrival, event_present,
                query_times, return_details=False):
        """Model inputs (contracts.MODEL_INPUT_KEYS). return_details gives ModelOutput whose
        calibration is per query [B,Q,3,6] and accepted marks queries with any slot [B,Q]."""
        if event_features.ndim != 5 or event_features.shape[2:] != (NUM_HANDS, NUM_JOINTS, FEATURE_CHANNELS):
            raise ValueError('Expected event features [B,E,2,21,14]')
        if event_valid.shape != event_features.shape[:-1] or query_times.ndim != 2:
            raise ValueError('Invalid event/query dimensions')
        events = make_events(event_features, event_valid, event_camera, event_capture, event_arrival, event_present)
        pose, params, rows = self.decode_queries(events, self.encode_events(events), query_times.float())
        return ModelOutput(pose, params, rows) if return_details else pose


class LiteStream(EventStream):
    """HandLite event by event: each event is encoded once (geometry and hand tokens);
    a query runs slot selection, calibrator, corrected anchor and decoder. Equals forward()."""

    def to_absolute(self, encoded, origin):
        return replace(encoded, capture=encoded.capture.double()+origin, stamp=encoded.stamp.double()+origin)

    def decode(self, time):
        encoded = self.store.encoded.map(lambda value: value[None])
        encoded = replace(encoded, capture=(encoded.capture-time).float(), stamp=(encoded.stamp-time).float())
        query = torch.zeros(1, 1, device=self.device)
        return self.model.decode_queries(relative_events(self.store.raw, time), encoded, query)[0][0, 0]


In [ ]:
%%writefile /content/gigahands_training/hand_tracking/lite_export.py
"""Export HandLite's three fixed-shape networks for lite_runtime: ONNX (onnxruntime) and
ncnn (converted by pnnx). Shapes have no batch axis in ncnn; ONNX graphs take a batch of one.

    encoder     features [2, 21*14], camera one-hot [3]                  -> tokens [P, dim]
    calibrator  tokens [P*3K, dim], pool bias [3, P*3K]                  -> correction [3, 6]
    decoder     tokens [P*3K, dim], token validity 0/1 [P*3K], gaps [P*3K], anchor features [42, A] -> offset [42, 3]
P = config.event_tokens: 10 (one per finger and hand) or 2 (one per hand); A = config.anchor_features.
"""
from dataclasses import asdict
import contextlib
import json
import os
from pathlib import Path
import torch
from torch import nn
from .constants import NUM_CAMERAS, NUM_HANDS, HAND_JOINTS
from .lite import HAND_FEATURES
from .lite_runtime import EXPORT_FORMAT, GRAPHS, META_FILE

FORMATS = ('onnx', 'ncnn')


class _Graph(nn.Module):
    def __init__(self, module):
        super().__init__()
        self.module = module


class EncoderGraph(_Graph):
    def forward(self, features, camera):
        return self.module(features, camera)


class CalibratorGraph(_Graph):
    def forward(self, tokens, pool_bias):
        return self.module(tokens, pool_bias)


class DecoderGraph(_Graph):
    def forward(self, tokens, token_valid, gaps, anchor_features):
        return self.module(tokens, token_valid, gaps, anchor_features)


def graphs(model):
    """name -> (module, example inputs with a batch of one)."""
    model = model.eval()
    dim, tokens = model.config.dim, model.config.event_tokens*model.slots
    torch.manual_seed(0)
    token_example = torch.randn(1, tokens, dim)
    result = dict(encoder=(EncoderGraph(model.encoder), (torch.randn(1, NUM_HANDS, HAND_FEATURES),
                                                         torch.eye(NUM_CAMERAS)[:1])),
                  decoder=(DecoderGraph(model.decoder), (token_example, torch.ones(1, tokens), torch.rand(1, tokens),
                                                         torch.randn(1, HAND_JOINTS, model.config.anchor_features))))
    if model.calibrator is not None:
        result['calibrator'] = (CalibratorGraph(model.calibrator), (token_example, torch.zeros(1, NUM_CAMERAS, tokens)))
    return result


@contextlib.contextmanager
def _working_directory(path):
    previous = Path.cwd()
    os.chdir(path)
    try:
        yield
    finally:
        os.chdir(previous)


def export_lite(model, directory, formats=FORMATS):
    """Write the graphs and lite.json (config) to a new directory; returns the metadata."""
    unknown = set(formats)-set(FORMATS)
    if unknown:
        raise ValueError(f'Unknown formats {sorted(unknown)}')
    directory = Path(directory)
    directory.mkdir(parents=True)
    exported = graphs(model)
    if 'calibrator' not in exported:
        # The runtime always loads three graphs; without a head it never calls this one.
        exported['calibrator'] = (CalibratorGraph(_ZeroCalibration()), (torch.zeros(1, 1, model.config.dim),
                                                                          torch.zeros(1, NUM_CAMERAS, 1)))
    return write_graphs(exported, directory, formats, GRAPHS, META_FILE,
                        dict(architecture='lite', export_format=EXPORT_FORMAT, config=asdict(model.config)))


def write_graphs(exported, directory, formats, names, meta_file, meta):
    """Export exported[name] = (module, example inputs) for names to ONNX and/or ncnn in the
    (new) directory, and write meta plus the formats and graph input shapes to meta_file."""
    with torch.no_grad():
        for name in names:
            module, example = exported[name]
            if 'onnx' in formats:
                program = torch.onnx.export(module, example, dynamo=True, verbose=False)
                program.save(str(directory/f'{name}.onnx'))
            if 'ncnn' in formats:
                _pnnx(module, example, directory, name)
    meta = dict(meta, formats=list(formats), graphs={
        name: dict(inputs=[list(value.shape[1:]) for value in exported[name][1]]) for name in names})
    (directory/meta_file).write_text(json.dumps(meta, indent=2), encoding='utf-8')
    return meta


class _ZeroCalibration(nn.Module):
    def forward(self, tokens, pool_bias):
        return pool_bias[:, :, :1].expand(-1, -1, 6)*0.


def _pnnx(module, example, directory, name):
    """Convert with pnnx and keep only the ncnn param/bin (pnnx writes next to its input)."""
    import pnnx
    with _working_directory(directory):
        pnnx.export(module, f'{name}.pt', example)
        intermediate = [path for path in Path('.').glob(f'{name}*')
                        if not path.name.endswith(('.ncnn.param', '.ncnn.bin', '.onnx'))
                        or path.name.endswith('.pnnx.onnx')]
        intermediate += list(Path('__pycache__').glob(f'{name}*'))
        for path in intermediate:
            path.unlink()
        cache = Path('__pycache__')
        if cache.is_dir() and not any(cache.iterdir()):
            cache.rmdir()


In [ ]:
%%writefile /content/gigahands_training/hand_tracking/lite_runtime.py
"""HandLite deployment runtime without PyTorch: numpy geometry around the three exported
networks, run by ncnn or ONNX Runtime. Mirrors hand_tracking.lite.LiteStream.

    runtime = LiteRuntime('exports/lite_model', backend='ncnn')
    runtime.push(camera, features [2,21,14], valid [2,21], capture_time, arrival_time)
    pose = runtime.query(time)          # [2,21,3] world units

Times are absolute seconds (float64). Needs numpy plus `ncnn` or `onnxruntime`.
"""
from dataclasses import dataclass
import json
from pathlib import Path
import numpy as np
from .config import saved_config
from .constants import (NUM_CAMERAS, NUM_HANDS, NUM_JOINTS, HAND_JOINTS, MODEL_CHANNELS, UV, RAY_ORIGIN,
                        RAY_DIRECTION, DELAY, TIME_UNIT_S, MISS_SCALE, ARRIVAL_TOLERANCE_S, REPEAT_TOLERANCE_S,
                        MIN_TIME_STD_S, MAX_EXTRAPOLATION_S, MIN_RAY_SIN2, MIN_RAY_DEPTH, RAY_RESIDUAL_FLOOR, OUTLIER_MARGIN,
                        MASK_OFF,
                        STREAM_KEEP_MARGIN_S,
                        check_event)

META_FILE = 'lite.json'
# Graph interface version; 2: the decoder takes token validity (0/1), not an additive bias;
# 3: the encoder makes config.event_tokens tokens per event (finger tokens). Version 2
# exports load with the options they predate off (config.LEGACY_OPTIONS); 4: the decoder
# takes config.anchor_features per joint (single-ray anchor flag).
EXPORT_FORMAT = 4
READABLE_FORMATS = (2, 3, 4)
GRAPHS = ('encoder', 'calibrator', 'decoder')


# --------------------------------------------------------------------------- geometry

def rodrigues(w):
    """Rotation vectors [...,3] -> matrices [...,3,3]."""
    theta = np.linalg.norm(w, axis=-1)[..., None, None]
    small = theta < 1e-4
    safe = np.where(small, 1., theta)
    a = np.where(small, 1-theta**2/6, np.sin(safe)/safe)
    half = safe/2
    b = np.where(small, .5-theta**2/24, .5*(np.sin(half)/half)**2)
    x, y, z = w[..., 0], w[..., 1], w[..., 2]
    zero = np.zeros_like(x)
    skew = np.stack((zero, -z, y, z, zero, -x, -y, x, zero), -1).reshape(*w.shape[:-1], 3, 3)
    return np.eye(3)+a*skew+b*(skew@skew)


def triangulate(origins, directions, mask):
    """origins/directions [...,C,3], mask [...,C] -> point [...,3], valid [...] (see geometry.triangulate)."""
    mask = (mask & np.isfinite(origins).all(-1) & np.isfinite(directions).all(-1)
            & (np.linalg.norm(directions, axis=-1) > 1e-8))
    d = np.where(mask[..., None], directions, 0.)
    o = np.where(mask[..., None], origins, 0.)
    d = d/np.maximum(np.linalg.norm(d, axis=-1, keepdims=True), 1e-8)
    # sum over rays of (I - d d^T) and (I - d d^T) o; masked rays have d = o = 0.
    count = mask.sum(-1)
    a = count[..., None, None]*np.eye(3)-np.einsum('...ci,...cj->...ij', d, d)
    rhs = (o-d*(d*o).sum(-1, keepdims=True)).sum(-2)
    cameras = d.shape[-2]
    sin2 = np.zeros(count.shape)
    for i in range(cameras):
        for j in range(i+1, cameras):
            sin2 = np.maximum(sin2, np.square(np.cross(d[..., i, :], d[..., j, :])).sum(-1))
    valid = (count >= 2) & (sin2 > MIN_RAY_SIN2)
    a = np.where(valid[..., None, None], a, np.eye(3))
    r0, r1, r2 = a[..., 0, :], a[..., 1, :], a[..., 2, :]
    c0, c1, c2 = np.cross(r1, r2), np.cross(r2, r0), np.cross(r0, r1)
    point = (rhs[..., 0:1]*c0+rhs[..., 1:2]*c1+rhs[..., 2:3]*c2)/(r0*c0).sum(-1, keepdims=True)
    depth = ((point[..., None, :]-o)*d).sum(-1)
    valid = valid & ((depth > 0) | ~mask).all(-1)
    return np.where(valid[..., None], point, 0.), valid


def ray_residuals(point, origins, directions):
    """Angular residual of each ray [...,C] to point [...,3] (see geometry.ray_residuals)."""
    d = directions/np.maximum(np.linalg.norm(directions, axis=-1, keepdims=True), 1e-8)
    relative = point[..., None, :]-origins
    depth = (relative*d).sum(-1)
    return np.linalg.norm(relative-depth[..., None]*d, axis=-1)/np.maximum(depth, 1e-6)


def robust_triangulate(origins, directions, mask, outlier_ratio):
    """triangulate dropping at most one outlier ray (see geometry.robust_triangulate);
    returns point, valid and the rays used."""
    point, valid = triangulate(origins, directions, mask)
    cameras = np.arange(mask.shape[-1])
    several = mask.sum(-1) >= 3
    best, second = np.zeros(valid.shape), np.zeros(valid.shape)
    best_point, best_valid, dropped = point, valid, np.zeros_like(mask)
    for k in cameras:
        keep = mask & (cameras != k)
        loo_point, loo_valid = triangulate(origins, directions, keep)
        residual = ray_residuals(loo_point, origins, directions)
        spread = np.where(keep, residual, 0.).max(-1)
        ratio = np.where(loo_valid & mask[..., k] & several, residual[..., k]/np.maximum(spread, RAY_RESIDUAL_FLOOR), 0.)
        better = ratio > best
        second = np.where(better, best, np.maximum(second, ratio))
        best = np.where(better, ratio, best)
        best_point = np.where(better[..., None], loo_point, best_point)
        best_valid = np.where(better, loo_valid, best_valid)
        dropped = np.where(better[..., None], cameras == k, dropped)
    suspect = best > outlier_ratio
    ambiguous = suspect & (best < OUTLIER_MARGIN*second)
    reject = suspect & ~ambiguous
    valid = np.where(reject, best_valid, valid) & ~ambiguous
    point = np.where(valid[..., None], np.where(reject[..., None], best_point, point), 0.)
    return point, valid, mask & ~(reject[..., None] & dropped)


def calibrate(origins, directions, params):
    """Rays [...,3,2,21,3] corrected by params [3,6] (see geometry.apply_calibration)."""
    rotation = rodrigues(params[:, :3])                                            # [3,3,3]
    return origins+params[:, None, None, 3:], np.einsum('cxy,...chjy->...chjx', rotation, directions)


def sample(origins, directions, mask, capture, params=None, outlier_ratio=0.):
    """Rays [...,3,2,21,3] of the three cameras (optionally corrected by params [3,6]) ->
    point [...,2,21,3], ok [...,2,21], stamp (mean capture of the used rays) [...,2,21];
    outlier_ratio > 0 drops one inconsistent ray (robust_triangulate)."""
    if params is not None:
        origins, directions = calibrate(origins, directions, params)
    # [...,camera,hand,joint,xyz] -> [...,hand,joint,camera,xyz]
    rays = list(range(origins.ndim-4))+[origins.ndim-3, origins.ndim-2, origins.ndim-4, origins.ndim-1]
    masks = list(range(mask.ndim-3))+[mask.ndim-2, mask.ndim-1, mask.ndim-3]
    used = np.transpose(mask, masks)
    if outlier_ratio:
        point, ok, used = robust_triangulate(np.transpose(origins, rays), np.transpose(directions, rays), used,
                                             outlier_ratio)
    else:
        point, ok = triangulate(np.transpose(origins, rays), np.transpose(directions, rays), used)
    weight = used.astype(np.float64)
    stamp = (capture[..., None, None, :]*weight).sum(-1)/np.maximum(weight.sum(-1), 1)
    return point, ok, stamp


def anchor_at(point, ok, stamp, query, lookback_s, hold_s, single=None):
    """Anchor [2,21,3] and flags [2,21,5] at query from slot samples [S,...] in arrival
    order (all arrived by the query); see events.query_anchor. single = (origin, direction,
    has, capture, reference, known) [2,21,...] adds single-ray anchors and a sixth flag."""
    s = len(point)
    if not s:
        # No slot in the span (e.g. a camera outage): origin and all flags off, as in torch.
        return np.zeros((NUM_HANDS, NUM_JOINTS, 3)), np.zeros((NUM_HANDS, NUM_JOINTS, 5+(single is not None)))
    index = np.arange(s)
    now = ok[-1]
    tau = stamp-query
    usable = ok & (-tau <= hold_s)
    latest = np.where(usable, index[:, None, None], -1).max(0)
    joint = latest >= 0
    pick = np.maximum(latest, 0)
    hands, joints = np.indices(joint.shape)
    anchor = point[pick, hands, joints]
    age = np.where(joint, query-stamp[pick, hands, joints], 0.)
    moving = np.zeros_like(joint)
    if lookback_s > 0:
        use = usable & (-tau <= lookback_s)
        same = (np.abs(stamp[:, None]-stamp[None]) <= REPEAT_TOLERANCE_S) & ok[:, None] & ok[None]
        count = (same & use[None]).sum(1)
        w = np.where(use, 1/np.maximum(count, 1), 0.)
        total = w.sum(0)
        mean_tau = (w*tau).sum(0)/np.maximum(total, 1e-12)
        mean_point = (w[..., None]*point).sum(0)/np.maximum(total, 1e-12)[..., None]
        centred = tau-mean_tau
        var_tau = (w*centred**2).sum(0)
        slope = (w[..., None]*centred[..., None]*(point-mean_point)).sum(0)/np.maximum(var_tau, 1e-12)[..., None]
        moving = (total >= 2-1e-6) & (var_tau > MIN_TIME_STD_S**2*total)
        extrapolated = mean_point+slope*np.clip(-mean_tau, 0., MAX_EXTRAPOLATION_S)[..., None]
        anchor = np.where(moving[..., None], extrapolated, anchor)
    extra = []
    if single is not None:
        anchor, joint, moving, age, ray = single_ray_anchor(anchor, joint, moving, age, now, query, *single)
        extra = [ray]
    weight = joint[..., None].astype(np.float64)
    hand_mean = (anchor*weight).sum(-2, keepdims=True)/np.maximum(weight.sum(-2, keepdims=True), 1)
    hand = np.broadcast_to(joint.any(-1, keepdims=True), joint.shape)
    anchor = np.where(joint[..., None], anchor, hand_mean)
    flags = np.stack((now, joint, hand, age/TIME_UNIT_S, moving, *extra), -1).astype(np.float64)
    return anchor, flags


def single_ray_anchor(anchor, joint, moving, age, now, query, origin, direction, has, capture, reference, known):
    """See events.single_ray_anchor."""
    reference = np.where(joint[..., None], anchor, reference)
    known = joint | known
    weight = known[..., None].astype(np.float64)
    hand = (reference*weight).sum(-2, keepdims=True)/np.maximum(weight.sum(-2, keepdims=True), 1)
    reference = np.where(known[..., None], reference, hand)
    known = np.broadcast_to(known.any(-1, keepdims=True), known.shape)
    direction = direction/np.maximum(np.linalg.norm(direction, axis=-1, keepdims=True), 1e-12)
    relative = reference[..., None, :]-origin
    along = (relative*direction).sum(-1)                                                # [2,21,C]
    residual = np.linalg.norm(relative-along[..., None]*direction, axis=-1)/np.maximum(along, 1e-6)
    best = np.where(has, residual, np.inf).argmin(-1)[..., None]
    pick = lambda value: np.take_along_axis(value, best, -1)[..., 0]
    origin, direction = (np.take_along_axis(value, best[..., None], -2)[..., 0, :] for value in (origin, direction))
    ray = has.any(-1) & ~now & known
    depth = np.maximum(pick(along), MIN_RAY_DEPTH)[..., None]
    anchor = np.where(ray[..., None], origin+direction*depth, anchor)
    age = np.where(ray, query-pick(capture), age)
    return anchor, joint | ray, moving & ~ray, age, ray


def joint_features(raw, valid, point, ok):
    """Event features [2,21,11], valid, nominal sample -> encoder input [2,21,14]."""
    origin, direction = raw[..., RAY_ORIGIN], raw[..., RAY_DIRECTION]
    relative = point-origin
    miss = relative-direction*(relative*direction).sum(-1, keepdims=True)
    miss = np.where((valid & ok)[..., None], miss*MISS_SCALE, 0.)
    x = np.concatenate((raw[..., UV]*2-1, origin, direction, raw[..., DELAY]/TIME_UNIT_S, miss,
                        valid[..., None], (ok & valid)[..., None]), -1)
    return np.where(valid[..., None], x, 0.)


def select_slot_events(events, time, per_camera, span_s):
    """Each camera's per_camera latest events (with .camera/.capture/.arrival, in arrival
    order) arrived by time and captured within span_s, in arrival order; None for padding
    (first). The runtimes' events.select_slots."""
    chosen = [[] for _ in range(NUM_CAMERAS)]
    for index in range(len(events)-1, -1, -1):
        event = events[index]
        if event.arrival <= time+ARRIVAL_TOLERANCE_S and event.capture >= time-span_s \
                and len(chosen[event.camera]) < per_camera:
            chosen[event.camera].append(index)
    ordered = sorted(i for indices in chosen for i in indices)
    return [None]*(NUM_CAMERAS*per_camera-len(ordered))+[events[i] for i in ordered]


# --------------------------------------------------------------------------- backends

class OnnxGraphs:
    def __init__(self, directory, threads=1, names=GRAPHS):
        import onnxruntime
        options = onnxruntime.SessionOptions()
        if threads:
            options.intra_op_num_threads = threads
        self.sessions = {name: onnxruntime.InferenceSession(str(Path(directory)/f'{name}.onnx'), options,
                                                            providers=['CPUExecutionProvider'])
                         for name in names}

    def run(self, name, *arrays):
        session = self.sessions[name]
        feed = {value.name: np.asarray(array, np.float32)[None] for value, array in zip(session.get_inputs(), arrays)}
        return session.run(None, feed)[0][0]


class NcnnGraphs:
    """ncnn nets. Inputs are copied into ncnn-owned Mats and kept alive until extract():
    the Python binding crashes when an in-place layer writes into numpy-backed memory."""

    def __init__(self, directory, threads=1, fp16=True, names=GRAPHS):
        import ncnn
        self.ncnn = ncnn
        self.nets = {}
        for name in names:
            net = ncnn.Net()
            net.opt.use_vulkan_compute = False
            for option in ('use_fp16_packed', 'use_fp16_storage', 'use_fp16_arithmetic'):
                setattr(net.opt, option, fp16)
            if threads:
                net.opt.num_threads = threads
            prefix = str(Path(directory)/name)
            if net.load_param(prefix+'.ncnn.param') or net.load_model(prefix+'.ncnn.bin'):
                raise RuntimeError(f'Cannot load ncnn graph {prefix}')
            self.nets[name] = net

    def run(self, name, *arrays):
        mats = [self.ncnn.Mat(np.ascontiguousarray(array, np.float32)).clone() for array in arrays]
        extractor = self.nets[name].create_extractor()
        for i, mat in enumerate(mats):
            extractor.input(f'in{i}', mat)
        status, output = extractor.extract('out0')
        if status:
            raise RuntimeError(f'ncnn {name} failed with {status}')
        return np.array(output).copy()


# --------------------------------------------------------------------------- runtime

@dataclass
class _Event:
    camera: int
    capture: float
    arrival: float
    origin: np.ndarray     # [3,2,21,3] each camera's latest ray at this arrival
    direction: np.ndarray
    mask: np.ndarray       # [3,2,21]
    ray_capture: np.ndarray  # [3]
    evident: bool
    point: np.ndarray      # [2,21,3] nominal triangulation of those rays
    ok: np.ndarray         # [2,21]
    stamp: np.ndarray      # [2,21] mean capture time of the rays used
    tokens: np.ndarray     # [event_tokens,dim]
    raw: np.ndarray        # [2,21,11], invalid joints zeroed
    valid: np.ndarray      # [2,21]


class LiteRuntime:
    def __init__(self, directory, backend='ncnn', threads=1, fp16=True, graphs=None):
        """graphs: any object with run(name, *arrays), instead of loading a backend."""
        directory = Path(directory)
        meta = json.loads((directory/META_FILE).read_text(encoding='utf-8'))
        if meta.get('architecture') != 'lite':
            raise ValueError(f'{directory} is not a HandLite export')
        if meta.get('export_format') not in READABLE_FORMATS:
            raise ValueError(f'{directory} was exported for another runtime version; re-run python -m training.export')
        self.config = saved_config('lite', meta['config'])
        if graphs is not None:
            self.graphs = graphs
        elif backend == 'ncnn':
            self.graphs = NcnnGraphs(directory, threads, fp16)
        elif backend == 'onnxruntime':
            self.graphs = OnnxGraphs(directory, threads)
        else:
            raise ValueError("backend must be 'ncnn' or 'onnxruntime'")
        self.keep_s = self.config.context_s+STREAM_KEEP_MARGIN_S
        self.reset()

    def reset(self):
        self.events = []
        self.latest = [None]*NUM_CAMERAS          # newest accepted event of each camera
        self.last_arrival = None                  # kept even when every event is pruned

    def __len__(self):
        return len(self.events)

    def push(self, camera, features, valid, capture_time, arrival_time):
        """Encode one arrived camera frame; False (ignored) for a capture no newer than
        that camera's latest. Frames must arrive in order; frames without detections count."""
        camera = check_event(camera, np.shape(features), np.shape(valid))
        capture, arrival = float(capture_time), float(arrival_time)
        if self.last_arrival is not None and arrival < self.last_arrival:
            raise ValueError('Events must be pushed in arrival order')
        self.last_arrival = arrival
        if self.latest[camera] is not None and capture <= self.latest[camera].capture:
            return False
        valid = np.array(valid, dtype=bool, copy=True)      # stored: callers may reuse their buffer
        raw = np.where(valid[..., None], np.asarray(features, np.float64)[..., :MODEL_CHANNELS], 0.)
        origin = np.zeros((NUM_CAMERAS, NUM_HANDS, NUM_JOINTS, 3))
        direction = np.zeros_like(origin)
        mask = np.zeros((NUM_CAMERAS, NUM_HANDS, NUM_JOINTS), bool)
        ray_capture = np.zeros(NUM_CAMERAS)
        newest = list(self.latest)
        newest[camera] = _Event(camera, capture, arrival, None, None, None, None, False, None, None, None, None, raw, valid)
        for c, event in enumerate(newest):
            if event is not None and event.capture >= arrival-self.config.sample_max_age_s:
                origin[c], direction[c] = event.raw[..., RAY_ORIGIN], event.raw[..., RAY_DIRECTION]
                mask[c], ray_capture[c] = event.valid, event.capture
        point, ok, stamp = sample(origin, direction, mask, ray_capture)
        features = joint_features(raw, valid, point, ok).reshape(NUM_HANDS, -1)
        tokens = self.graphs.run('encoder', features, np.eye(NUM_CAMERAS)[camera])
        event = _Event(camera, capture, arrival, origin, direction, mask, ray_capture, bool((ok & valid).any()),
                       point, ok, stamp, tokens, raw, valid)
        self.latest[camera] = event
        self.events.append(event)
        while self.events and self.events[0].capture < arrival-self.keep_s:
            self.events.pop(0)
        return True

    def slots(self, time):
        """See select_slot_events."""
        return select_slot_events(self.events, time, self.config.slots_per_camera, self.config.event_span_s)

    def _single_rays(self, latest, params, time):
        """anchor_at's single-ray inputs (see HandLite.single_rays): each camera's corrected
        ray of each joint in the latest slot event's table, and the joint's newest nominal
        triangulation captured within anchor_depth_memory_s."""
        origin, direction = calibrate(latest.origin, latest.direction, params)
        joints_last = lambda value: np.moveaxis(value, 0, 2)                                    # [2,21,3(,3)]
        hands, joints = np.indices((NUM_HANDS, NUM_JOINTS))
        stored = [e for e in self.events if e.arrival <= time+ARRIVAL_TOLERANCE_S]
        ok = np.stack([e.ok for e in stored]) & (np.stack([e.stamp for e in stored]) >= time-self.config.anchor_depth_memory_s)
        last = np.where(ok, np.arange(len(stored))[:, None, None], -1).max(0)
        reference = np.stack([e.point for e in stored])[np.maximum(last, 0), hands, joints]
        return (joints_last(origin), joints_last(direction), joints_last(latest.mask),
                np.broadcast_to(latest.ray_capture, (NUM_HANDS, NUM_JOINTS, NUM_CAMERAS)), reference, last >= 0)

    def query(self, time):
        """Pose [2,21,3] at time (not before the latest arrival)."""
        pose, _ = self.query_details(time)
        return pose

    def query_details(self, time):
        """Pose [2,21,3] and the camera correction used [3,6]."""
        time = float(time)
        if self.last_arrival is None:
            raise ValueError('No events yet')
        if time < self.last_arrival:
            raise ValueError('Query before the latest arrival')
        slots = self.slots(time)
        dim, per = self.config.dim, self.config.event_tokens
        present = np.array([event is not None for event in slots])
        tokens = np.stack([event.tokens if event is not None else np.zeros((per, dim)) for event in slots])
        tokens = tokens.reshape(-1, dim)
        cameras = np.array([event.camera if event is not None else -1 for event in slots])
        token_valid = np.repeat(present, per)
        token_camera = np.repeat(cameras, per)
        params = np.zeros((NUM_CAMERAS, 6))
        if self.config.calibration_head:
            pool = np.where((token_camera[None] == np.arange(NUM_CAMERAS)[:, None]) & token_valid[None], 0., MASK_OFF)
            evident = np.array([event is not None and event.evident for event in slots])
            informative = ((cameras[None] == np.arange(NUM_CAMERAS)[:, None]) & evident[None]).any(-1)
            params = np.where(informative[:, None], self.graphs.run('calibrator', tokens, pool), 0.)
        used = [event for event in slots if event is not None]
        if used:
            point, ok, stamp = sample(np.stack([e.origin for e in used]), np.stack([e.direction for e in used]),
                                      np.stack([e.mask for e in used]), np.stack([e.ray_capture for e in used]), params,
                                      self.config.ray_outlier_ratio)
        else:
            point = np.zeros((0, NUM_HANDS, NUM_JOINTS, 3))
            ok = np.zeros((0, NUM_HANDS, NUM_JOINTS), bool)
            stamp = np.zeros((0, NUM_HANDS, NUM_JOINTS))
        single = self._single_rays(used[-1], params, time) if used and self.config.anchor_ray_depth else None
        anchor, flags = anchor_at(point, ok, stamp, time, self.config.anchor_lookback_s, self.config.anchor_hold_s,
                                  single)
        # Without any slot there is no ray: the single-ray flag is off, as in torch.
        flags = np.pad(flags, ((0, 0), (0, 0), (0, self.config.anchor_features-3-flags.shape[-1])))
        capture = np.array([event.capture if event is not None else time for event in slots])
        gaps = np.repeat(np.where(present, np.maximum(time-capture, 0.)/TIME_UNIT_S, 0.), per)
        features = np.concatenate((anchor, flags), -1).reshape(HAND_JOINTS, -1)
        offset = self.graphs.run('decoder', tokens, token_valid, gaps, features)
        return anchor+offset.reshape(NUM_HANDS, NUM_JOINTS, 3), params


In [ ]:
%%writefile /content/gigahands_training/hand_tracking/objectives.py
"""Training losses (pose, bone length, calibration auxiliary) and evaluation totals."""
from dataclasses import dataclass
import math
import torch
import torch.nn.functional as F


def calibration_loss(prediction, target, rotation_deg=5., position_std=.1):
    """Mean squared calibration error in prior-std units over cameras with a known target.

    prediction/target [N,3,6]; target rows of NaN (e.g. real captures, padding) are ignored.
    The common rig motion is unobservable, so the optimum is its posterior mean.
    """
    scale = prediction.new_tensor([math.radians(rotation_deg)]*3+[position_std]*3)
    known = torch.isfinite(target).all(-1)
    error = ((prediction-torch.where(known[...,None], target, 0.))/scale).square().mean(-1)
    return (error*known).sum()/known.sum().clamp_min(1)


BONES = [(0,1+4*f) for f in range(5)] + [(1+4*f+j,2+4*f+j) for f in range(5) for j in range(3)]


def pose_loss(prediction, target, target_mask, world_unit_cm, bone_weight=.1, beta_m=.006):
    """Masked SmoothL1 plus target-specific bone length loss, in meters.

    prediction/target [B,...,2,21,3]; world_unit_cm [B] converts each sample, so
    beta and the loss scale do not depend on the export's world unit.
    """
    valid = target_mask.bool()
    scale = (world_unit_cm/100).reshape(-1, *[1]*(prediction.ndim-1))
    prediction = prediction.float()*scale
    safe_target = torch.where(valid[...,None], target, 0.)*scale
    joints = F.smooth_l1_loss(prediction, safe_target, beta=beta_m, reduction='none').mean(-1)
    position = (joints*valid).sum()/valid.sum().clamp_min(1)
    a, b = map(list, zip(*BONES))
    pred_lengths = (prediction[...,a,:]-prediction[...,b,:]).norm(dim=-1)
    true_lengths = (safe_target[...,a,:]-safe_target[...,b,:]).norm(dim=-1)
    bone_mask = valid[...,a] & valid[...,b]
    bone = ((pred_lengths-true_lengths).abs()*bone_mask).sum()/bone_mask.sum().clamp_min(1)
    return position+bone_weight*bone


def error_totals(prediction, target, mask, world_unit_cm):
    """MPJPE numerator (mm), PCK@20mm numerator and valid joint count for [B,2,21,3]."""
    valid = mask.bool()
    distance = (prediction-torch.where(valid[...,None],target,0.)).norm(dim=-1)
    mm = distance*world_unit_cm[:,None,None]*10
    return (mm*valid).sum(), ((mm<20)&valid).sum(), valid.sum()


@dataclass
class PoseTotals:
    """Running MPJPE and PCK@20mm over the final query of each window (the reported metric),
    against the target frame's targets, and MPJPE against the world-frame targets."""
    error_mm: float = 0.
    correct: int = 0
    joints: int = 0
    samples: int = 0
    world_error_mm: float = 0.

    def add(self, pose, batch):
        """pose [B,Q,2,21,3] against a batch's target/target_mask/world_unit_cm."""
        with torch.no_grad():
            error, correct, joints = error_totals(pose[:,-1].float(), batch['target'][:,-1],
                                                  batch['target_mask'][:,-1], batch['world_unit_cm'])
            world, _, _ = error_totals(pose[:,-1].float(), batch['target_world'][:,-1], batch['target_mask'][:,-1],
                                       batch['world_unit_cm'])
        self.error_mm += error.item()
        self.world_error_mm += world.item()
        self.correct += correct.item()
        self.joints += joints.item()
        self.samples += len(pose)

    @property
    def mpjpe_mm(self):
        return self.error_mm/self.joints if self.joints else float('nan')

    def summary(self):
        if not self.joints:
            raise ValueError('No valid observations/targets in this loader')
        return dict(mpjpe_mm=self.mpjpe_mm, pck20=self.correct/self.joints, mpjpe_world_mm=self.world_error_mm/self.joints,
                    samples=self.samples, joints=self.joints)


@dataclass
class WeightedMean:
    total: float = 0.
    weight: float = 0.

    def add(self, value, weight):
        self.total += value*weight
        self.weight += weight

    @property
    def mean(self):
        """None when nothing was weighted (e.g. no known calibration target)."""
        return self.total/self.weight if self.weight else None


In [ ]:
%%writefile /content/gigahands_training/hand_tracking/stream.py
"""Arrival-ordered event buffering for streaming inference, separate from model math.

EventStream owns pushed frames, drops stale captures and prunes old events; each model's
stream subclass says how its per-event encoding moves between relative and absolute time
(to_absolute) and how a query is decoded from the stored events (decode)."""
from dataclasses import dataclass
from typing import Any, NamedTuple
import torch
from .contracts import RawEvents
from .constants import STREAM_KEEP_MARGIN_S, check_event
from .events import relative_events


class PendingEvent(NamedTuple):
    features: torch.Tensor
    valid: torch.Tensor
    camera: int
    capture: float
    arrival: float


@dataclass
class StreamCache:
    """Stored events (absolute float64 capture and arrival times) and their encodings:
    any object with map(fn) and append(other, dim)."""
    raw: RawEvents
    encoded: Any

    def select(self, keep):
        return StreamCache({key: value[keep] for key, value in self.raw.items()},
                           self.encoded.map(lambda value: value[keep]))


class EventStream:
    """Encode each accepted camera event once; query any time from the latest arrival on.

    Events must arrive in order. Empty detections replace older camera rays; duplicate or
    older captures are dropped, matching the batch event policy (events.accepted_captures).
    push() owns its input tensors. query()/flush() encode all pending events in one batch.
    """

    def __init__(self, model):
        self.model = model.eval()
        self.config = model.config
        self.device = next(model.parameters()).device
        # Pruning keeps a margin beyond the exact context a new event can reach.
        self.keep_s = self.config.context_s + STREAM_KEEP_MARGIN_S
        self.reset()

    def reset(self):
        self.store: StreamCache | None = None
        self.pending: list[PendingEvent] = []
        self.newest_capture: dict[int, float] = {}

    def __len__(self):
        stored = 0 if self.store is None else len(self.store.raw['camera'])
        return stored + len(self.pending)

    def _latest_arrival(self):
        if self.pending:
            return self.pending[-1].arrival
        return None if self.store is None else float(self.store.raw['arrival'][-1])

    def push(self, camera, features, valid, capture_time, arrival_time):
        """Own a frame; return False when its capture does not advance that camera."""
        arrival_time = float(arrival_time)
        capture_time = float(capture_time)
        camera = check_event(camera, torch.as_tensor(features).shape, torch.as_tensor(valid).shape)
        latest = self._latest_arrival()
        if latest is not None and arrival_time < latest:
            raise ValueError('Events must be pushed in arrival order')
        if capture_time <= self.newest_capture.get(camera, float('-inf')):
            return False
        event = PendingEvent(torch.as_tensor(features, dtype=torch.float32, device=self.device).clone(),
                             torch.as_tensor(valid, dtype=torch.bool, device=self.device).clone(),
                             camera, capture_time, arrival_time)
        self.newest_capture[camera] = capture_time
        self.pending.append(event)
        return True

    def _pending_raw(self):
        return RawEvents(
            features=torch.stack([event.features for event in self.pending]),
            valid=torch.stack([event.valid for event in self.pending]),
            camera=torch.tensor([event.camera for event in self.pending], device=self.device),
            capture=torch.tensor([event.capture for event in self.pending], dtype=torch.float64, device=self.device),
            arrival=torch.tensor([event.arrival for event in self.pending], dtype=torch.float64, device=self.device))

    @torch.inference_mode()
    def flush(self):
        """Encode pending events transactionally: a failure leaves the buffer intact."""
        if not self.pending:
            return
        origin = self.pending[-1].arrival
        raw = self._pending_raw()
        old = self.store
        if old is not None:
            keep = old.raw['capture'] >= origin-self.keep_s
            if not keep.all():
                old = old.select(keep)
            raw = {key: torch.cat((old.raw[key], value)) for key, value in raw.items()}
        count = len(raw['camera'])
        targets = torch.arange(count-len(self.pending), count, device=self.device)
        encoded = self.model.encode_events(relative_events(raw, origin), targets).map(lambda value: value[0])
        encoded = self.to_absolute(encoded, origin)
        if old is not None:
            encoded = old.encoded.append(encoded, dim=0)
        self.store = StreamCache(raw, encoded)
        self.pending.clear()

    @torch.inference_mode()
    def query(self, time):
        """Pose [2,21,3] at time (not before the latest arrival)."""
        time = float(time)
        latest = self._latest_arrival()
        if latest is None:
            raise ValueError('No events yet')
        if time < latest:
            raise ValueError('Query before the latest arrival')
        self.flush()
        return self.decode(time)

    def to_absolute(self, encoded, origin):
        """A new unbatched encoding with its times on origin -> absolute float64 times."""
        return encoded

    def decode(self, time):
        raise NotImplementedError


In [ ]:
# 5. 데이터 분리·파일 크기와 소스 확인
import importlib
importlib.invalidate_caches()
from hand_tracking.checkpoints import build_model
from hand_tracking.config import config_class
from hand_tracking.data import read_manifest
SOURCE_HASHES = {'training/__init__.py': '7c8c03e7f7bc8c274605e0bb27d727a603f1df54d45069d05dcf3032015bc8c2', 'training/train.py': 'bcca19786cf60523eb42cfdff26c056df0753a372cbeed0d49c7d8fc481c94c6', 'training/evaluate.py': '57286460cb6a59873bf64c5dc74f70c32198ed0a473397a06af65fd74a5b543a', 'training/predict.py': '944c6c495aaabd3945eb93a971737567071b5b6e55db55524f20c1a09c3785a6', 'training/export.py': 'fa195693f72b8f2317d05fc6a5fa9f2091f45f26253ea5baf147db824edce7e6', 'hand_tracking/__init__.py': '166e0ecdd3de289e24bb3e1b27966969d9e37e6ae1e6599069c69d4ce2b7c07e', 'hand_tracking/checkpoints.py': 'fcfa456c67932ea9ca68608d4db8c01046ad0e05b7a55664db6c0e9cbceb90c0', 'hand_tracking/config.py': 'bd1c9bccde2351a3504939ceff9b74676589bd53e8e56cbe2abd0dd33563c4bc', 'hand_tracking/constants.py': '99a3ee30af690d4f872b7c6bc24ea0ec19ea072e4c9632a45f209847ea581cd5', 'hand_tracking/contracts.py': 'a315fe6eb2f67f9af38b07a6a2653294124288eca4af4ee50d3b1d4671484417', 'hand_tracking/data.py': '94665a759782efab604694ee7e9f60b06b0250c4b062e070a56c11dd8060d635', 'hand_tracking/direct.py': 'b62fec70c57abbfe699839a096ab0e8ea49c13dcd58b2cae5fbc6c31df6ce0d8', 'hand_tracking/direct_export.py': '78c8f02b3d1c417f21ff33448dc099d9fae4d383773994b12f3c840ba776bd67', 'hand_tracking/direct_runtime.py': 'e9664f33fac921cbfd97f8ae43434ca948e0003f314e142aaeeecd75a74337fe', 'hand_tracking/engine.py': '1cf8748e36e546285b0936ca71f51054997812a3549080dfc478022a97d4779d', 'hand_tracking/events.py': '9323d3429e7e1502d280c77c478698fc9074f589b923774f8d2a7bc2a6dde996', 'hand_tracking/geometry.py': '089e0c25eb6a79e084033d779905bf81b469c44f850a1b9945182f17ee8cc644', 'hand_tracking/layers.py': 'b223f991eb047405c468d9a27a905f04fd1cffeab7acdf60e3ce8ca735dee939', 'hand_tracking/lite.py': 'ea68ed3498bdcb1052f96d4402118bd8a92cfef8311a71dbb6d227d26eaa84a3', 'hand_tracking/lite_export.py': '07040319c7349070b7ec1073cba22ffeb9c38e583e6208f99c90a2ad4fe42078', 'hand_tracking/lite_runtime.py': '3fece16ed649d260a02f5944ea81f2c9a7d07dfc8330e87459c4a35f2650359f', 'hand_tracking/objectives.py': 'b0ce28ba3b2abd8f19432b7c931c3df3739c138d65bcd3aa7382a267fc315265', 'hand_tracking/stream.py': 'f588c248fe519786661a157e06353a7f29d9b14a7a329490665335b998faf9c0'}
for name, expected in SOURCE_HASHES.items():
    actual = hashlib.sha256((WORK_DIR / name).read_text().encode()).hexdigest()
    assert actual == expected, f'소스 변경 감지: {name}. 변경 실험은 노트북과 실행 이름을 새로 만드세요.'
rows = read_manifest(DATA_DIR)  # 참가자/원본 train-val 중복이면 오류
for row in tqdm(rows, desc='Check dataset'):
    assert (DATA_DIR / row['file']).stat().st_size == row['bytes'], row['file']
for split in ('train', 'val'):
    selected = [r for r in rows if r['split'] == split]
    print(split, 'clips:', len(selected), 'participants:', len({r['participant'] for r in selected}))
cfg = config_class(ARCH)(dropout=DROPOUT, **{key[2:].replace('-', '_'): value for key, value in MODEL_OPTIONS.items()})
temporary_model = build_model(ARCH, cfg)
print('parameters:', f'{sum(p.numel() for p in temporary_model.parameters()):,}')
del temporary_model

def run_python(module, *arguments):
    command = [sys.executable, '-u', '-m', module, *map(str, arguments)]
    print(' '.join(command), flush=True)
    process = subprocess.Popen(command, cwd=WORK_DIR, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
        code = process.wait()
        if code:
            raise subprocess.CalledProcessError(code, command)
    except BaseException:
        if process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
        raise


def run_training(arguments, epochs, batch_size, done=0):
    """run_python('training.train') with progress bars read from its log: all epochs, and
    this epoch's train/val batches (moved every 50 batches, as train.py logs; the batch totals
    are upper bounds, so a bar fills at the phase's end). Other lines print as usual."""
    import math, re
    command = [sys.executable, '-u', '-m', 'training.train', *map(str, arguments)]
    print(' '.join(command), flush=True)
    overall = tqdm(total=epochs, initial=done, desc='전체', unit='epoch')
    phases = {name: tqdm(total=1, desc=f'{name} (epoch {done+1})', unit='batch') for name in ('train', 'val')}
    sizes = {}
    def finish(bar):
        if bar.n < bar.total:
            bar.update(bar.total-bar.n)
    process = subprocess.Popen(command, cwd=WORK_DIR, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            if sizes_line := re.search(r'train_windows/epoch<=([\d,]+) val_windows<=([\d,]+)', line):
                for name, windows in zip(phases, sizes_line.groups()):
                    sizes[name] = max(math.ceil(int(windows.replace(',', ''))/batch_size), 1)
                    phases[name].reset(total=sizes[name])
                print(line, end='', flush=True)
            elif progress := re.match(r'(train|val) batch=(\d+) samples=\d+ MPJPE=([\d.]+)mm(.*)', line):
                name, batches = progress[1], int(progress[2])
                if name == 'val':
                    finish(phases['train'])
                bar = phases[name]
                bar.update(min(batches, bar.total)-bar.n)
                bar.set_postfix_str(f'MPJPE {progress[3]}mm {progress[4].strip()}'.strip())
            elif ended := re.match(r'epoch=(\d+)/\d+ (.*)', line):
                epoch = int(ended[1])
                overall.update(epoch-overall.n)
                overall.set_postfix_str(ended[2].strip())
                print(line, end='', flush=True)
                for name, bar in phases.items():   # ready for the next epoch
                    if epoch < epochs:
                        bar.reset(total=sizes.get(name, 1))
                        bar.set_description(f'{name} (epoch {epoch+1})')
                    else:
                        finish(bar)
            else:
                print(line, end='', flush=True)
        code = process.wait()
        if code:
            raise subprocess.CalledProcessError(code, command)
    except BaseException:
        if process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
        raise
    finally:
        for bar in (overall, *phases.values()):
            bar.close()


## GPU 동작 확인
본학습 전에 실제 데이터의 작은 배치로 CUDA·mixed precision·역전파·체크포인트 저장을 확인합니다. 이 결과는 성능 평가가 아니며 본학습 가중치로 사용하지 않습니다. GPU 메모리가 부족하면 배치 크기를 줄이고 다시 실행하세요.


In [ ]:
# 6. GPU smoke test — 본학습 실행 폴더와 분리
import uuid
smoke_dir = WORK_DIR / ('smoke_' + uuid.uuid4().hex[:8])
run_python('training.train',
           '--data', DATA_DIR, '--output', smoke_dir, '--epochs', 1,
           '--batch-size', min(BATCH_SIZE, 2), *MODEL_ARGS, '--calibration-weight', CALIBRATION_WEIGHT,
           '--max-clips', 4, '--max-train-batches', 2, '--max-val-batches', 2, '--train-queries', TRAIN_QUERIES,
           '--windows-per-clip', 2, '--val-windows-per-clip', 2,
           '--workers', 0, '--device', 'cuda')
print('GPU smoke test passed.')


## 본학습 / 재개
- 기본값은 학습 클립마다 epoch당 최대 16개 무작위 윈도우입니다. 윈도우는 query 시각 16개와, 첫 query보다 약 0.75초(direct)·1.45초(lite) 전부터 마지막 query까지 도착한 이벤트(최대 128개)입니다.
- 검증 구간은 고정됩니다. MPJPE는 마지막 query의 3D 관절 오차(mm)이며 작을수록 좋습니다. 기본 정답 좌표계는 `rig`(정해 둔 카메라 배치를 실제로 놓인 대로 본 좌표계, 관측할 수 없는 rig 전체 어긋남 제거)이고, 참 월드 좌표 오차는 `mpjpe_world_mm`으로 함께 기록합니다.
- 카메라 보정은 최근 약 0.5초의 이벤트에서만 추정합니다(장기 기억 없음; lite는 카메라당 최근 이벤트 8개).
- 학습 증강은 없습니다. 카메라 배치가 고정이므로 그 배치의 월드 좌표를 그대로 배우게 합니다. 설치 오차는 데이터 생성기가 클립마다 따로 넣습니다.
- 보조 정답 `calibration_target[3,6]`은 카메라마다 회전 보정 벡터(rad) 3개와 위치 보정(world) 3개입니다. 모델 입력에는 넣지 않고 `CALIBRATION_WEIGHT`로 보조 손실을 조절합니다. 회전 벡터의 세 번째 성분은 카메라의 roll 각도와 다르므로 roll 0°에서도 0이 아닐 수 있습니다.
- **매 epoch 완료 시** Drive에 `last.pt`, 개선 시 `best.pt`를 저장합니다. 런타임이 끊기면 완료되지 않은 epoch는 다시 학습합니다.
- anchor는 이벤트마다 카메라별 최신 ray를 삼각측량한 표본에 직선을 맞춰 query 시각으로 외삽합니다. ray 궤적 직접 풀이와 잔차 게이트는 시뮬레이션 데이터에서 정확도를 낮춰 기본으로 끕니다. 이전 구조의 가중치는 쓸 수 없으니 새 실행으로 학습하세요.
- 같은 설정·같은 `RUN_NAME`으로 다시 실행하면 마지막 완료 epoch부터 재개합니다. 재개 중 배치 크기·epoch 수·모델 설정을 바꾸면 거부됩니다. 설정 변경은 새 `RUN_NAME`으로 시작하세요.
- 첫 epoch가 끝나기 전에 중단되어 `last.pt`가 없으면 새 `RUN_NAME`을 사용하세요. 기존 결과를 자동 삭제하지 않습니다.


In [ ]:
# 7. Train / Resume — Drive에 epoch마다 저장
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
source_file = EXPERIMENT_DIR / 'source_hashes.json'
if source_file.is_file():
    assert json.loads(source_file.read_text()) == SOURCE_HASHES, '이 실행의 모델 소스가 달라졌습니다. 새 RUN_NAME을 사용하세요.'
else:
    source_file.write_text(json.dumps(SOURCE_HASHES, indent=2))
    snapshot = EXPERIMENT_DIR / 'source_snapshot'
    snapshot.mkdir(exist_ok=True)
    for name in SOURCE_HASHES:
        (snapshot / name).parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(WORK_DIR / name, snapshot / name)

if not CACHE_DIR.exists():
    assert shutil.disk_usage('/content').free > 14003487420, '전처리 캐시(데이터의 약 2배) 공간이 부족합니다.'
resume = (RUN_DIR / 'last.pt').is_file()
if RUN_DIR.exists() and not resume:
    raise RuntimeError('완료된 체크포인트가 없는 기존 실행 폴더입니다. RUN_NAME을 새 이름으로 바꾸고 설정 셀부터 다시 실행하세요.')
arguments = [
    '--data', DATA_DIR, '--output', RUN_DIR, '--epochs', EPOCHS,
    '--batch-size', BATCH_SIZE, *MODEL_ARGS, '--lr', LEARNING_RATE, '--calibration-weight', CALIBRATION_WEIGHT,
    '--windows-per-clip', WINDOWS_PER_CLIP, '--val-windows-per-clip', VAL_WINDOWS_PER_CLIP, '--train-queries', TRAIN_QUERIES,
    '--workers', WORKERS, '--threads', 2, '--seed', SEED, '--device', 'cuda', '--cache-dir', CACHE_DIR,
] + (['--compile'] if COMPILE else [])
done = 0
if resume:
    arguments.append('--resume')
    print('Resume:', RUN_DIR / 'last.pt')
    done = len({json.loads(line)['epoch'] for line in (RUN_DIR / 'metrics.jsonl').read_text().splitlines() if line.strip()})
run_training(arguments, EPOCHS, BATCH_SIZE, done)
print('체크포인트:', RUN_DIR / 'best.pt')


In [ ]:
# 8. 학습 곡선
import matplotlib.pyplot as plt
records = [json.loads(line) for line in (RUN_DIR / 'metrics.jsonl').read_text().splitlines() if line.strip()]
# 중단 시 마지막 완료 epoch의 기록이 반복될 수 있어 epoch별 마지막 행 표시
records = list({r['epoch']: r for r in records}.values())
records.sort(key=lambda r: r['epoch'])
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for split in ('train', 'val'):
    axes[0].plot([r['epoch'] for r in records], [r[split]['mpjpe_mm'] for r in records], label=split)
    axes[1].plot([r['epoch'] for r in records], [r[split]['pck20'] * 100 for r in records], label=split)
    axes[2].plot([r['epoch'] for r in records], [r[split]['calibration_mse'] for r in records], label=split)
axes[0].set(title='MPJPE', xlabel='Epoch', ylabel='mm')
axes[1].set(title='PCK @ 20 mm', xlabel='Epoch', ylabel='%')
axes[2].set(title='Calibration auxiliary loss', xlabel='Epoch', ylabel='Normalized MSE')
for ax in axes:
    ax.grid(alpha=.3)
    ax.legend()
fig.tight_layout()
fig.savefig(EXPERIMENT_DIR / 'learning_curves.png', dpi=150)
plt.show()


## 검증: 학습 모델 vs 삼각측량 기준선
동일한 검증 윈도우에서 비교합니다. 기본은 모든 검증 클립의 최대 4개 윈도우입니다. `EVAL_WINDOWS_PER_CLIP=0`으로 바꾸면 전체 검증 윈도우를 평가하므로 시간이 더 걸립니다. 기준선은 학습 없이 nominal 캘리브레이션으로 만든 삼각측량 anchor입니다. 검증은 모델 선택에 사용되었으므로 독립된 최종 test 성능은 아닙니다.


In [ ]:
# 9. 같은 조건의 모델/기준선 평가
EVAL_WINDOWS_PER_CLIP = 4
assert (RUN_DIR / 'best.pt').is_file(), '먼저 학습 체크포인트를 생성하세요.'
eval_dir = EXPERIMENT_DIR / ('eval_' + uuid.uuid4().hex[:8])
eval_dir.mkdir()
common = ['--data', DATA_DIR, '--windows-per-clip', EVAL_WINDOWS_PER_CLIP,
          '--batch-size', BATCH_SIZE, '--workers', WORKERS, '--device', 'cuda']
run_python('training.evaluate', '--triangulation-only', '--output', eval_dir / 'anchor.json', *common)
run_python('training.evaluate', '--checkpoint', RUN_DIR / 'best.pt', '--output', eval_dir / 'model.json', *common)
anchor_report = json.loads((eval_dir / 'anchor.json').read_text())
model_report = json.loads((eval_dir / 'model.json').read_text())
assert anchor_report['samples'] == model_report['samples']
print(f"Anchor MPJPE: {anchor_report['mpjpe_mm']:.2f} mm")
print(f"Model  MPJPE: {model_report['mpjpe_mm']:.2f} mm")
print(f"Reduction: {anchor_report['mpjpe_mm'] - model_report['mpjpe_mm']:.2f} mm (양수이면 개선)")
print('결과 저장:', eval_dir)


In [ ]:
# 10. 검증 클립 1개 예측 및 3D 비교
chosen = next(row for row in rows if row['split'] == 'val' and row['windows'] > 0)
prediction_file = EXPERIMENT_DIR / ('prediction_' + uuid.uuid4().hex[:8] + '.npz')
run_python('training.predict', '--checkpoint', RUN_DIR / 'best.pt',
           '--input', DATA_DIR / chosen['file'], '--output', prediction_file)
with np.load(prediction_file, allow_pickle=False) as data:
    predicted = data['predicted_xyz_cm'].copy()
    target = data['target_xyz'] * float(data['world_unit_cm'])
    valid = data['target_mask'].copy()
from hand_tracking.objectives import BONES
fig = plt.figure(figsize=(8, 7))
ax = fig.add_subplot(111, projection='3d')
for hand, color in enumerate(('tab:blue', 'tab:orange')):
    for a, b in BONES:
        if valid[hand, a] and valid[hand, b]:
            ax.plot(*target[hand, [a,b]].T, color=color, linestyle='--', alpha=.6)
        ax.plot(*predicted[hand, [a,b]].T, color=color)
ax.set(xlabel='X (cm)', ylabel='Y (cm)', zlabel='Z (cm)', title='Prediction: solid / Ground truth: dashed')
combined = np.concatenate((predicted.reshape(-1,3), target[valid]), axis=0)
center = (combined.min(0)+combined.max(0))/2
radius = max(float(np.ptp(combined,axis=0).max())/2, 1.)
ax.set_xlim(center[0]-radius,center[0]+radius)
ax.set_ylim(center[1]-radius,center[1]+radius)
ax.set_zlim(center[2]-radius,center[2]+radius)
ax.set_box_aspect((1,1,1))
plt.show()
print('원본:', chosen['source'])
print('저장:', prediction_file)


## 배포용 내보내기 (선택)
`training/export.py`가 신경망을 **ncnn과 ONNX**로 내보내고(direct: encoder·query 2개, lite: encoder·보정기·decoder 3개), 라즈베리 파이용 런타임(`hand_tracking/direct_runtime.py` 또는 `lite_runtime.py`)과 PyTorch 스트림의 차이를 검증 클립에서 확인합니다. 결과 폴더 전체를 장치로 복사하면 PyTorch 없이 `numpy`와 `ncnn`만으로 추론합니다.


In [ ]:
# 11. 배포용 내보내기 + 런타임 확인
needed = ('onnx', 'onnxscript', 'onnxruntime', 'ncnn', 'pnnx')
missing = [name for name in needed if importlib.util.find_spec(name) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)
export_target = EXPERIMENT_DIR / ('deploy_' + uuid.uuid4().hex[:8])
run_python('training.export', '--checkpoint', RUN_DIR / 'best.pt', '--output', export_target,
           '--check-input', DATA_DIR / chosen['file'])
print('내보내기:', export_target)


## 저장 위치 및 다시 시작하기
- `MyDrive/GigaHands/runs/<RUN_NAME>/checkpoints/best.pt`: 검증 오차가 가장 낮은 가중치
- `last.pt`: 마지막 완료 epoch의 모델·optimizer·scheduler·난수 상태
- `run.json`, `metrics.jsonl`: 설정과 학습 기록
- 상위 실행 폴더: 소스 사본, 학습 곡선, 기준선 비교, 예측 결과, 배포용 모델(`deploy_*/` 또는 `model_*.onnx`)

Colab 재연결 시 위에서부터 실행하고 같은 `RUN_NAME`과 설정을 사용하세요. GPU smoke test는 생략할 수 있습니다. 7번 셀이 `last.pt`를 찾아 자동 재개합니다.

**실행 검증 범위:** 노트북 구조·내장 코드·데이터 검사·학습/재개/평가/추론 파이프라인은 로컬 CPU로 검증했습니다. 이 파일을 만들 때 실제 Colab GPU 세션에서 전체 학습을 수행한 것은 아닙니다. Colab에서 6번 GPU smoke test로 CUDA 경로를 먼저 확인하세요.
